# Level 6 — Knowledge-only — COCO_2014 ONLY

Notebook này kế thừa Level 5A nhưng chạy **ablation chỉ dùng nhánh tri thức**.

## Cấu hình chính

- Dataset: **MS COCO 2014 / Karpathy split**
- Visual contribution = **0**
- Knowledge contribution = **1** (`g = 1`)
- Tổng knowledge memory luôn = **5 tokens**
- Dùng **tối đa 3 relation RelTR thật**
- Nếu thiếu relation, số token còn thiếu được bù bằng caption retrieval
- Không dùng YOLO
- Không duplicate relation
- Không tạo relation giả
- Không train lại; dùng checkpoint Level 5A tương ứng seed 42–46

## Chính sách composition

- Có từ 3 relation trở lên → **2 caption + 3 relation**
- Có 2 relation → **3 caption + 2 relation**
- Có 1 relation → **4 caption + 1 relation**
- Có 0 relation → **5 caption**

Mỗi relation là một SPO đầy đủ `(subject, predicate, object)` và được serialize thành text để dùng cùng text encoder 512d.

**Output bắt buộc:**

`/content/drive/MyDrive/DataKhoaLuan/Data/baseline_level6/_Chi_Dung_TriThuc`

Kho tri thức Level 6 được materialize dưới:

`DataKhoaLuan/Data/baseline_level6/_Chi_Dung_TriThuc/knowledge_base_level6/`

Notebook không dùng COCO_2017 hoặc Flickr cho retrieval, inference hay evaluation.


# Level 5A — train độc lập 5 seed, kiến trúc 5 tokens

Không có kết quả thực nghiệm dựng sẵn trong notebook này. Giữ RelTR → Semantic/Jaccard → Min–Max → MMR,
gate, attention và beam k=3 từ mã nguồn đã cung cấp. Model có `Linear(512,256)` và 5 knowledge tokens.

**Cài một lần:** giải nén `Level5A_5Tokens_Train5Seeds.zip` trên máy, tải cả thư mục
`multiseed_5tokens_v1` vào `/content/drive/MyDrive/Data/Data/baseline_level5/lv5_A/`.
Mở notebook này trên Colab GPU. Giữ các shared_modules và dữ liệu có sẵn.

**Luồng chạy:** `prepare` với seed42 một lần → mỗi seed42–46 chạy `train` → `retune` → `test`
→ sau cùng `aggregate`. Không dùng Run all để tự động đổi phase.

Mỗi seed train lại từ **cùng bản L4 gốc Lan01_seed45**, optimizer/scheduler/RNG mới.
Resume chỉ dùng checkpoint của chính seed đó. Retune/Test nạp model L5 đã train, không dùng L4 thay thế.
Recipe theo phân công: 15 epochs, batch64, LR5e-5; không early stopping. Giữ w=0.6 và grid lambda
{0.5,0.6,0.7,0.8} của nhánh 5-token hiện tại; prepare chọn lambda train trên Val một lần cho cả nhóm.

Kết quả mới: `lv5_A/trained_5tokens_v1/seed42` ... `seed46`.
Kết quả cũ ở `l4_5tokens_eval` không được đưa vào mean/std mới.

Resume train lưu mỗi 250 batch và cuối epoch: có thể phải tính lại tối đa phần chưa lưu;
caption lưu mỗi 64 ảnh; bộ chấm lưu từng metric đã tính xong. Không thể cứu dữ liệu chưa kịp lên Drive.
Chỉ một phiên Colab ghi vào một seed tại một thời điểm. Không xóa thư mục khi muốn resume.


## 1 Chuẩn bị Colab và mount Drive
Chạy từ mục này khi runtime đã bị mất. Không ghi lại shared modules.


In [1]:
# Level 5A gốc: chỉ lưu để đối chiếu; không chạy trong Level 6.
if globals().get("RUN_LEVEL5_ORIGINAL", False):
    import os, sys, subprocess, hashlib
    from pathlib import Path
    from google.colab import drive

    sys.dont_write_bytecode = True
    drive.mount('/content/drive')
    DATA_ROOT = Path('/content/drive/MyDrive/DataKhoaLuan/Data')
    BASE_LV5 = DATA_ROOT / 'baseline_level5'

    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nltk==3.9.1', 'pycocoevalcap==1.2', 'faiss-cpu'], check=True)
    java_home = Path('/usr/lib/jvm/java-11-openjdk-amd64')
    if not (java_home / 'bin/java').is_file():
        subprocess.run(['apt-get', 'update', '-qq'], check=True)
        subprocess.run(['apt-get', 'install', '-y', '-qq', 'openjdk-11-jre-headless'], check=True)
    assert (java_home / 'bin/java').is_file(), 'Cần Java 11 để giữ môi trường evaluator L4'
    os.environ['JAVA_HOME'] = str(java_home)
    os.environ['PATH'] = str(java_home / 'bin') + ':' + os.environ.get('PATH', '')
    subprocess.run(['java', '-version'], check=True)

    import nltk
    assert nltk.__version__ == '3.9.1', 'NLTK đã nạp phiên bản khác; restart runtime rồi chạy lại'
    for corpus in ['wordnet', 'omw-1.4']:
        assert nltk.download(corpus, quiet=True, raise_on_error=True)
    for module_name in ['multiseed','storage','level5a_pipeline','level5_model','semantic_expansion','mmr_diversity']:
        sys.modules.pop(module_name, None)  # re-import the current files from Drive; no file mutation
    print('Môi trường sẵn sàng; chưa tạo hoặc ghi file trong baseline_level5.')


## 2 Kiểm tra đúng bộ mã và checkpoint gốc
Dừng nếu thiếu file; không tự tạo dữ liệu đầu vào.


In [2]:
# Level 5A gốc: chỉ lưu để đối chiếu; không chạy trong Level 6.
if globals().get("RUN_LEVEL5_ORIGINAL", False):
    from pathlib import Path
    import sys
    DATA_ROOT = Path('/content/drive/MyDrive/DataKhoaLuan/Data')
    BASE_LV5 = DATA_ROOT / 'baseline_level5'
    DIR_LV5_A = BASE_LV5 / 'lv5_A'
    DIR_SHARED_MODULES = BASE_LV5 / 'shared_modules'
    CODE_DIR = DIR_LV5_A / 'multiseed_5tokens_v1'
    INIT_CKPT_PATH = DATA_ROOT / 'baseline_level4/full/Lan01_seed45/best_model.pt'
    for directory in [DIR_LV5_A, BASE_LV5/'lv5_B_Yolo', DIR_SHARED_MODULES, CODE_DIR]:
        assert directory.is_dir(), f'Thiếu thư mục có sẵn: {directory}'
    for name in ['multiseed.py','storage.py','level5a_pipeline.py','level5_model.py',
                 'coco_eval_worker.py','assignment_baselines.json']:
        assert (CODE_DIR/name).is_file(), f'Thiếu mã: {CODE_DIR/name}'
    assert INIT_CKPT_PATH.is_file(), f'Thiếu L4: {INIT_CKPT_PATH}'
    for directory in [DIR_SHARED_MODULES, CODE_DIR]:
        if str(directory) in sys.path:
            sys.path.remove(str(directory))
        sys.path.insert(0, str(directory))
    import multiseed as MS
    import level5a_pipeline as P
    import level5_model as M
    import storage
    import semantic_expansion
    import mmr_diversity
    for module in [MS, P, M, storage]:
        assert Path(module.__file__).resolve().parent == CODE_DIR.resolve(), (
            'Đang nạp bản mã khác. Restart runtime rồi chạy lại từ mục 1.'
        )
    for module in [semantic_expansion, mmr_diversity]:
        assert Path(module.__file__).resolve().parent == DIR_SHARED_MODULES.resolve()
    assert M.MEMORY_CONTRACT['knowledge_memory_tokens'] == 5
    expand_labels = semantic_expansion.expand_labels
    jaccard_semantic = semantic_expansion.jaccard_semantic
    mmr_select = mmr_diversity.mmr_select
    cosine_sim_vec = mmr_diversity.cosine_sim_vec
    print('Mã đúng nguồn:', CODE_DIR)
    print('Kiến trúc:', M.MEMORY_CONTRACT)


## 3 Chọn phase và seed

Lần đầu: `PHASE='prepare'`, `SEED=42`. Sau khi xong chuyển sang `train`, rồi `retune`, rồi `test`.
Sau Test42, đổi seed43 và bắt đầu từ `train`; tiếp tục đến seed46. Không cần prepare lại cho mỗi seed.
Mỗi lần đổi phase/seed, chạy lại mục **3 → 5 → 6 → 7**. Nếu mất runtime, chạy lại **1 → 7** cùng phase/seed.
Sau đủ 5 Test: đặt `PHASE='aggregate'`; mục 5 sẽ bỏ qua nạp model/dữ liệu lớn.


In [3]:
# Level 5A gốc: chỉ lưu để đối chiếu; không chạy trong Level 6.
if globals().get("RUN_LEVEL5_ORIGINAL", False):
    PHASE = 'test'  # prepare / train / retune / test / aggregate
    SEED = 44             # 42 / 43 / 44 / 45 / 46
    CONFIG = {
        'phase': PHASE, 'seed': SEED, 'w_fixed': 0.6,
        'data_root': str(DATA_ROOT),
        'vocab_path': None, 'annotations_dir': None,
        'feature_dir': None, 'kb_dir': None,
        'reltr_dir': str(DATA_ROOT / 'reltr_inputs'),
        'eval_timeout_seconds': 7200,
    }
    assert PHASE in {'prepare','train','retune','test','aggregate'}
    assert SEED in {42,43,44,45,46}
    assert PHASE != 'prepare' or SEED == 42
    print(CONFIG)


## 4 Kiểm tra shared modules
Giữ unit test của notebook cũ; các vector này không phải số liệu mô hình.


In [4]:
# Level 5A gốc: chỉ lưu để đối chiếu; không chạy trong Level 6.
if globals().get("RUN_LEVEL5_ORIGINAL", False):
    import json
    import numpy as np
    semantic_review = {}
    for label in ['dog','bicycle','person','car','airplane','cat','sports ball','cell phone','couch','bottle']:
        values = expand_labels([label])
        assert isinstance(values, set) and 0 < len(values) <= 8, f'Shared module không đạt giới hạn cho {label}'
        semantic_review[label] = sorted(values)
        print(f'[{label}] ({len(values)} mục): {sorted(values)}')
    fixture_vectors = np.asarray([[1,0], [1,0], [0,1], [0,-1], [-1,0], [1,0]], dtype=float)
    fixture_vectors = np.pad(fixture_vectors, ((0,0),(0,510)))  # 512d as in shared interface
    fixture_relevance = [1.0, 0.99, 0.9, 0.85, 0.8, 0.98]
    fixture_candidates = [dict(position=i, relevance=r, embedding=e) for i,(r,e) in enumerate(zip(fixture_relevance,fixture_vectors))]
    fixture_selected = mmr_select(fixture_candidates, k=5, lambda_param=0.5)
    assert len(fixture_selected) == len({c['position'] for c in fixture_selected}) == 5
    assert fixture_selected[0]['position'] == 0 and fixture_selected[1]['position'] != 1
    print('MMR UNIT TEST PASS:', [c['position'] for c in fixture_selected])
    from level5a_pipeline import write_json
    write_json(DIR_LV5_A / 'module_review' / 'semantic_10_labels.json', {
        'labels': semantic_review, 'nltk': nltk.__version__, 'wordnet': nltk.corpus.wordnet.get_version(),
        'test_type': 'real WordNet label expansion; not caption evaluation'
    })


## 5 Khởi tạo run và kiểm tra dữ liệu

Nạp dữ liệu và model vào runtime mới là bắt buộc; không đồng nghĩa train lại từ đầu.
Đổi seed tạo một run riêng. Cùng seed và cùng cấu hình sẽ tìm trạng thái đã lưu.
Nếu phiên bản thư viện hoặc input/mã đã đổi, code dừng để tránh ghép hai thí nghiệm khác nhau.


In [5]:
# Level 5A gốc: chỉ lưu để đối chiếu; không chạy trong Level 6.
if globals().get("RUN_LEVEL5_ORIGINAL", False):
    BRANCH = DIR_LV5_A / 'trained_5tokens_v1'
    if PHASE == 'aggregate':
        experiment = None
        print('Chỉ tổng hợp kết quả thật; không nạp model.')
    else:
        experiment = MS.Experiment(CONFIG)
        assert experiment.model.knowledge_proj.weight.shape == (256, 512)
        assert experiment.model.knowledge_memory_tokens == 5
        print('READY:', PHASE, '| seed:', SEED, '|', experiment.work)
        print('Init L4 SHA:', experiment.init_sha)
        print('Group SHA:', experiment.group_sha)


## 6 Thực thi đúng một phase

Train lưu hai slot checkpoint luân phiên, có checksum, optimizer/scheduler/RNG và vị trí batch.
Không có cờ reset tự động. Retune/Test yêu cầu train đủ 15 epoch và nạp trọng số L5 của chính seed đó.
Mất kết nối: chạy lại cùng phase/seed. Chạy lại phase đã hoàn tất sẽ dùng lại kết quả hợp lệ.


In [6]:
# Level 5A gốc: chỉ lưu để đối chiếu; không chạy trong Level 6.
if globals().get("RUN_LEVEL5_ORIGINAL", False):
    import errno
    import fcntl
    import hashlib
    import tempfile
    import time
    assert CONFIG['phase'] == PHASE and CONFIG['seed'] == SEED
    target = BRANCH if PHASE == 'aggregate' else experiment.work
    if experiment is not None:
        assert experiment.cfg == CONFIG and experiment.seed == SEED
    lock_name = hashlib.sha256(str(target).encode()).hexdigest()[:24]
    with (Path(tempfile.gettempdir()) / ('l5_run_' + lock_name + '.lock')).open('w') as handle:
        fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)
        try:
            for attempt in range(3):
                try:
                    if PHASE == 'aggregate':
                        MS.aggregate(BRANCH)
                    else:
                        storage.recover_json_tree(experiment.work)
                        storage.recover_json_tree(experiment.tuning)
                        experiment.run()
                    print('HOÀN TẤT:', PHASE, '| seed:', SEED)
                    break
                except OSError as exc:
                    transient = exc.errno in {errno.EIO,errno.ESTALE,errno.ETIMEDOUT,
                        errno.ENOTCONN,errno.ECONNRESET,errno.ECONNABORTED,errno.EAGAIN}
                    if not transient or attempt == 2:
                        raise
                    print('Lỗi I/O tạm thời; thử lại từ trạng thái đã lưu sau 5 giây:', exc)
                    time.sleep(5)
        except KeyboardInterrupt:
            print('Đã ngừng. Giữ nguyên thư mục và chạy lại cùng phase/seed để resume.')
            raise
        finally:
            fcntl.flock(handle, fcntl.LOCK_UN)


## 7 Xem kết quả đúng phase
Thiếu kết quả nghĩa là phase chưa hoàn tất hoặc Drive chưa truy cập được; không thay bằng 0.


In [7]:
# Level 5A gốc: chỉ lưu để đối chiếu; không chạy trong Level 6.
if globals().get("RUN_LEVEL5_ORIGINAL", False):
    import json
    if PHASE == 'prepare':
        path = BRANCH / 'prepare/selected_config.json'
    elif PHASE == 'aggregate':
        path = BRANCH / 'five_seed_summary.json'
    else:
        name = {'train':'training_log.json','retune':'validation_summary.json','test':'test_summary.json'}[PHASE]
        path = experiment.work / name
    print('File:', path)
    if not path.is_file():
        print('Chưa có kết quả hoàn tất. Xem toàn bộ output mục 6; không xóa thư mục.')
    else:
        result = storage.read_json(path)
        if PHASE in {'retune','test'}:
            print('Trained:', result['trained'], '| training seed:', result['training_seed'])
            print('Model L5 state SHA:', result['model_state_sha256'])
            print(json.dumps(result['metrics_x100'], indent=2))
            print('FPS:', result['online_fps'])
            print('FPS chỉ đo Semantic/Jaccard/MMR CPU; không gồm detector, FAISS hay decoder.')
        elif PHASE == 'train':
            print('Complete:', result['complete'], '| epochs:', result['completed_epochs'],
                  '| optimizer steps:', result['optimizer_steps'])
            print('Best epoch:', result['best_epoch'], '| best Val CIDEr raw:', result['best_val_cider'])
        elif PHASE == 'aggregate':
            print(json.dumps(result['comparison_to_reported_L4'], ensure_ascii=False, indent=2))
            print(result['limitation'])
        else:
            print(json.dumps(result, ensure_ascii=False, indent=2))


## Diễn giải so sánh với Level 4

Mean ± std được tính từ đúng 5 Test của các model đã train; std mẫu dùng `ddof=1`.
Code kiểm tra seed, hash model khác nhau, train hoàn tất, cùng dữ liệu/protocol và đúng khóa Test.
Model khác nhau không bắt buộc mọi điểm metric phải khác nhau; code không tạo ngẫu nhiên vào điểm số.

Mốc L4 đọc nguyên từ `assignment_baselines.json`: ví dụ CIDEr 110.25±0.52 là **mốc do nhóm cung cấp**,
không phải kết quả được notebook này tính lại. Chưa có 5 file L4 gốc nên chưa có so sánh ghép cặp hoặc kiểm định.
BLEU-2/3 L4 thiếu thì giữ N/A. Không so METEOR NLTK với METEOR COCO; không so FPS khác phạm vi đo.
Mức chênh lệch điểm là mô tả; không tự suy ra ý nghĩa thống kê hoặc bảo đảm L5 cải thiện.

Notebook L4 tham khảo dùng early stopping patience=4. L5 này giữ quy định train đủ 15 epoch trong phân công.
So sánh phản ánh toàn bộ phương án L5 có huấn luyện tiếp; chưa tách riêng tác dụng Semantic/MMR khỏi thêm training.


## Cách chạy khuyến nghị

Seed đầu tiên: chạy `/1 → /2 → /3 → /4 → /5 → /6 → /6A → /7 → /8 → /9`, sau đó chạy cell protocol check. Nếu không lỗi mới chạy `/10 → /11`.

`/4b` là optional và có thể bỏ qua.

Seed tiếp theo trong cùng runtime: đổi `SEED_TO_RUN`, rồi chạy `/3 → /11`.

Sau khi đủ seed 42–46 mới bật `RUN_AGGREGATE=True` và chạy `/12`.


## Level 6 / 1 — Mount Google Drive

Chạy từ đây nếu chỉ muốn phần Level 6. Không thực thi `MS.Experiment` hoặc phase train của notebook gốc.

### Hợp đồng thí nghiệm Level 6

Mỗi ảnh luôn có đúng **5 knowledge items**.

Ưu tiên dùng tối đa 3 relation RelTR thật sau dedup:

- `>= 3 relation` → `2 caption + 3 relation`
- `2 relation` → `3 caption + 2 relation`
- `1 relation` → `4 caption + 1 relation`
- `0 relation` → `5 caption`

Không duplicate relation, không tạo relation giả và không dùng YOLO.

Dataset contract: `DATASET_NAME = 'COCO_2014'`.
Mọi annotation/features/RelTR được dùng cho experiment phải thuộc protocol COCO_2014, không trộn COCO_2017 hoặc Flickr.


In [8]:
from google.colab import drive
RUN_LEVEL5_ORIGINAL = False
assert not RUN_LEVEL5_ORIGINAL
drive.mount('/content/drive')

Mounted at /content/drive


## Level 6 /2 — Runtime imports


In [9]:
import ast, csv, hashlib, importlib, importlib.metadata, inspect, json, math, os, re
import statistics, subprocess, sys, textwrap, time, traceback, types, unicodedata, zipfile
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import torch
sys.dont_write_bytecode = True
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'

def l6_sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def l6_now():
    return datetime.now(timezone.utc).isoformat()

## Level 6 /3 — Select one seed (42–46), full split, resume


In [10]:
# Change only SEED_TO_RUN to run the next independent seed.
SEED_TO_RUN = 42
FORCE_RERUN = False
FORCE_SOURCE_RESCAN = False  # True only when a fresh Drive inventory is needed.
EVAL_SPLIT = 'test'
DATASET_NAME = 'COCO_2014'
SEEDS = (42, 43, 44, 45, 46)
assert type(SEED_TO_RUN) is int and SEED_TO_RUN in SEEDS
assert EVAL_SPLIT in ('val', 'test')
assert DATASET_NAME == 'COCO_2014'
BASE_DIR = Path('/content/drive/MyDrive/DataKhoaLuan/Data')
LEVEL5_DIR = BASE_DIR / 'baseline_level5'
LEVEL6_DIR = BASE_DIR / 'baseline_level6/_Chi_Dung_TriThuc'
SEARCH_ROOT = Path('/content/drive')
DISCOVERY_ROOTS = (SEARCH_ROOT / 'MyDrive', SEARCH_ROOT / '.shortcut-targets-by-id')
L6_SOURCE_DIR_OVERRIDE = None
L6_SHARED_DIR_OVERRIDE = None
L6_CHECKPOINT_CANDIDATE_OVERRIDES = {}
L6_CONFIG = dict(
    experiment_name='Level6_KnowledgeOnly_COCO2014_Max3Relation_CaptionFallback',
    seed=SEED_TO_RUN, dataset_name=DATASET_NAME, split=EVAL_SPLIT,
    run_mode='one_seed_full_split', sample_limit=None,
    ablation_mode='knowledge_only',
    knowledge_token_count=5,
    target_caption_token_count=2,
    max_relation_token_count=3,
    relation_fallback='caption_fill_no_yolo',
    force_gate=1.0, visual_weight=0.0, knowledge_weight=1.0,
    beam_size=3, training=False
)
L6_CONTEXT = None
L6_RUN_STATE = None
KNOWLEDGE_BASE_NAME = 'knowledge_base_level6_max3relation_caption_fallback'
KNOWLEDGE_BASE_ROOT = LEVEL6_DIR / 'knowledge_base_level6'
KNOWLEDGE_BASE_ROOT.mkdir(parents=True, exist_ok=True)
L6_ADAPTER_VERSION = 'knowledge_only_max3relation_caption_fallback_g1_v5'
L6_DISALLOWED_DATASET_MARKERS = ('coco_2017', 'flickr', 'flickr30k')

def l6_assert_coco2014_only_path(path, role='input'):
    text = str(Path(path)).replace('\\', '/').casefold()
    bad = [marker for marker in L6_DISALLOWED_DATASET_MARKERS if marker in text]
    if bad:
        raise ValueError(f'{role} is outside COCO_2014-only protocol: {path} (matched {bad})')
    return Path(path)


def l6_output_path(relative):
    path = (LEVEL6_DIR / relative).resolve()
    if not path.is_relative_to(LEVEL6_DIR.resolve()):
        raise ValueError('Output outside Level6: ' + str(path))
    return path

def l6_seed_output(name):
    return l6_output_path(f'seed{SEED_TO_RUN}/{name}')

def l6_write_json(relative, value):
    path = l6_output_path(relative)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.pending')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    os.replace(temporary, path)
    return path

for folder in ('debug', 'cache', 'retrieval', 'summary', f'seed{SEED_TO_RUN}'):
    l6_output_path(folder).mkdir(parents=True, exist_ok=True)

def l6_log(message):
    text = str(message)
    print(text, flush=True)
    with l6_seed_output('run_log.txt').open('a', encoding='utf-8') as handle:
        handle.write(l6_now() + ' ' + text + '\n')

def l6_read_json_ro(path):
    # Unlike storage.read_json, never repairs/publishes a Level5 file.
    path = Path(path)
    problems = []
    for candidate in (path, path.with_name(path.name + '.pending')):
        try:
            value = json.loads(candidate.read_text(encoding='utf-8'))
            if candidate != path:
                l6_log('READ-ONLY pending copy: ' + str(candidate))
            return value
        except (OSError, UnicodeError, json.JSONDecodeError) as exc:
            problems.append(f'{candidate}: {exc}')
    raise FileNotFoundError('\n'.join(problems))


print(L6_CONFIG)
print('OUTPUT:', l6_seed_output(''))


{'experiment_name': 'Level6_KnowledgeOnly_COCO2014_Max3Relation_CaptionFallback', 'seed': 42, 'dataset_name': 'COCO_2014', 'split': 'test', 'run_mode': 'one_seed_full_split', 'sample_limit': None, 'ablation_mode': 'knowledge_only', 'knowledge_token_count': 5, 'target_caption_token_count': 2, 'max_relation_token_count': 3, 'relation_fallback': 'caption_fill_no_yolo', 'force_gate': 1.0, 'visual_weight': 0.0, 'knowledge_weight': 1.0, 'beam_size': 3, 'training': False}
OUTPUT: /content/drive/.shortcut-targets-by-id/1OCkkcQKlIYTCPLnGgb2y4aTf_VJFE7xQ/Data/baseline_level6/_Chi_Dung_TriThuc/seed42


## Level 6 /4 — Cached Source Audit

First run scans `MyDrive/DataKhoaLuan` and known DataKhoaLuan shortcut paths. Broad Drive roots are used only if required source modules are not found there.

Next runs restore `debug/source_audit.json`, verify Python source SHA256 and input path existence/size without directory traversal. No checkpoint, embedding or dataset is hashed in /4. Missing/changed inputs or `FORCE_SOURCE_RESCAN=True` trigger rescan. A valid legacy audit is upgraded without rescanning.

Changing seed does not invalidate this shared audit. Continue /3 → /4 → /5 → … → /11; /4b remains optional. To discover newly added files while all cached paths still exist, enable `FORCE_SOURCE_RESCAN`.


In [11]:
CORE_NAMES = ('multiseed.py', 'level5a_pipeline.py', 'level5_model.py',
              'coco_eval_worker.py', 'storage.py')
SHARED_NAMES = ('semantic_expansion.py', 'mmr_diversity.py')
REQUIRED_NAMES = CORE_NAMES + SHARED_NAMES

def l6_priority(path):
    parts = [part.casefold() for part in Path(path).parts]
    if any(parts[i:i+2] == ['baseline_level5', 'lv5_a'] for i in range(len(parts)-1)):
        return 0
    if 'baseline_level5' in parts:
        return 1
    return 2

def l6_choose_bundle(candidates, required, override=None):
    parents = set.intersection(*({Path(p).parent for p in candidates.get(name, [])}
                                for name in required))
    choices = sorted(parents, key=lambda p: (l6_priority(p), str(p)))
    result = dict(candidates=[str(p) for p in choices], selected=None, reason=None)
    if override is not None:
        requested = Path(override).resolve()
        if requested not in parents:
            result['reason'] = 'Override không phải bộ đầy đủ được tìm thấy trong phạm vi scan'
            return result
        result.update(selected=str(requested), reason='explicit_directory_override')
        return result
    if not choices:
        result['reason'] = 'Không có thư mục chứa đủ các module yêu cầu'
        return result
    best = [p for p in choices if l6_priority(p) == l6_priority(choices[0])]
    if len(best) != 1:
        result['reason'] = 'AMBIGUOUS: nhiều bộ cùng mức ưu tiên; cần chọn rõ thư mục'
        return result
    result.update(selected=str(best[0]), reason='unique_complete_bundle_at_highest_priority')
    return result

# Source Audit must not walk through huge image folders.
# COCO_2014 itself is the ONLY experiment dataset and is NOT excluded from the experiment.
# Only its large train2014/val2014/test2014 image subfolders are pruned during /4 source discovery.
L6_AUDIT_PRUNE_DIRS = {
    'coco_2017', 'flickr_image_dataset',
    'train2014', 'val2014', 'test2014',
    'train2017', 'val2017', 'test2017',
    '__pycache__', '.ipynb_checkpoints', '.git'
}

def l6_is_noncritical_scan_error(message):
    text = str(message).casefold()
    dataset_hints = (
        'coco_2014', 'coco_2017', 'flickr_image_dataset',
        'train2014', 'val2014', 'test2014',
        'train2017', 'val2017', 'test2017'
    )
    return any(hint in text for hint in dataset_hints)

def l6_critical_scan_errors(discovery):
    return [e for e in discovery.get('scan_errors', []) if not l6_is_noncritical_scan_error(e)]

def l6_scan_resources(root, output_root):
    root, output_root = Path(root).resolve(), Path(output_root).resolve()
    if not root.is_dir():
        raise FileNotFoundError(f'Chưa mount Drive hoặc thiếu search root: {root}')
    found = dict(sources={name: [] for name in REQUIRED_NAMES}, reltr=[],
                 checkpoints=[], caches=[], seed_directories=[],
                 trained_directories=[], scan_errors=[])
    def onerror(exc):
        found['scan_errors'].append(str(exc))
    for directory, dirs, files in os.walk(root, topdown=True, followlinks=False, onerror=onerror):
        here = Path(directory)
        kept = []
        for name in sorted(dirs):
            child = here / name
            resolved = child.resolve()
            if child.is_symlink() or not resolved.is_relative_to(root):
                continue
            if resolved == output_root or resolved.is_relative_to(output_root):
                continue
            if name.casefold() in L6_AUDIT_PRUNE_DIRS:
                continue
            kept.append(name)
            if name == 'trained_5tokens_v1':
                found['trained_directories'].append(str(resolved))
            if name in {f'seed{s}' for s in SEEDS}:
                found['seed_directories'].append(str(resolved))
        dirs[:] = kept
        for name in sorted(files):
            path = here / name
            if path.is_symlink():
                continue
            path = path.resolve()
            if not path.is_relative_to(root) or path.is_relative_to(output_root):
                continue
            lower = name.casefold()
            suffix = path.suffix.casefold()
            if name in found['sources']:
                found['sources'][name].append(str(path))
            if 'reltr' in lower and suffix == '.csv':
                found['reltr'].append(str(path))
            if suffix in ('.pt', '.pth', '.ckpt'):
                found['checkpoints'].append(str(path))
            cache_hint = any(word in str(path).casefold() for word in
                             ('cache', 'retrieval', 'embedding', 'feature', 'knowledge_base'))
            if (cache_hint and suffix in ('.npy', '.npz', '.json', '.jsonl', '.pkl', '.pickle', '.pt', '.pth', '.csv', '.index', '.faiss')) or lower in (
                'vocab.json', 'selected_config.json', 'assignment_baselines.json',
                'captions_karpathy_train.json', 'captions_karpathy_val.json', 'captions_karpathy_test.json'):
                found['caches'].append(str(path))
    return found

def l6_scan_drive_roots(roots, output_root):
    combined = dict(sources={name: [] for name in REQUIRED_NAMES}, reltr=[],
                    checkpoints=[], caches=[], seed_directories=[],
                    trained_directories=[], scan_errors=[], missing_search_roots=[])
    for root in roots:
        if not Path(root).is_dir():
            combined['missing_search_roots'].append(str(root))
            continue
        print('SEARCHING:', root, flush=True)
        found = l6_scan_resources(root, output_root)
        for name in REQUIRED_NAMES:
            combined['sources'][name].extend(found['sources'][name])
        for key in ('reltr', 'checkpoints', 'caches', 'seed_directories', 'trained_directories', 'scan_errors'):
            combined[key].extend(found[key])
    for name in REQUIRED_NAMES:
        combined['sources'][name] = sorted(set(combined['sources'][name]))
    for key in ('reltr', 'checkpoints', 'caches', 'seed_directories', 'trained_directories', 'scan_errors'):
        combined[key] = sorted(set(combined[key]))
    return combined


AUDIT_SCHEMA_VERSION = 2
AUDIT_MAX_SOURCE_BYTES = 2 * 1024 * 1024

def l6_audit_source_hash(path):
    path = Path(path)
    if path.suffix != '.py' or not path.is_file():
        raise ValueError('Missing Python source: ' + str(path))
    if path.stat().st_size > AUDIT_MAX_SOURCE_BYTES:
        raise ValueError('Python source exceeds audit size limit: ' + str(path))
    return l6_sha256(path)

def l6_audit_path(path):
    path = Path(path)
    resolved = path.resolve()
    if not resolved.is_relative_to(SEARCH_ROOT.resolve()) or resolved.is_relative_to(LEVEL6_DIR.resolve()):
        raise ValueError('Cached input path outside allowed Drive inputs: ' + str(path))
    return path

def l6_audit_inventory(discovery, previous=None):
    # Stat only: never read/hash checkpoints, arrays, features or datasets here.
    previous = previous or {}
    inventory = {}
    for kind in ('reltr', 'checkpoints', 'caches', 'trained_directories', 'seed_directories'):
        for value in discovery[kind]:
            path = l6_audit_path(value)
            directory = kind in ('trained_directories', 'seed_directories')
            if not (path.is_dir() if directory else path.is_file()):
                raise FileNotFoundError(f'Missing cached {kind} path: {path}')
            record = dict(path=value, kind='directory' if directory else 'file')
            if not directory:
                record['size_bytes'] = path.stat().st_size
                if value in previous and previous[value].get('size_bytes') != record['size_bytes']:
                    raise ValueError('Cached file size changed: ' + value)
            if kind in ('checkpoints', 'seed_directories'):
                seeds = re.findall(r'(?:^|[/\\])seed(\d+)(?=[/\\]|$)', value)
                record['seed_from_path'] = int(seeds[0]) if len(seeds) == 1 else None
                if len(seeds) > 1 or (seeds and int(seeds[0]) not in SEEDS):
                    raise ValueError('Ambiguous/unsupported cached seed path: ' + value)
                if kind == 'checkpoints' and path.suffix.lower() not in ('.pt','.pth','.ckpt'):
                    raise ValueError('Invalid checkpoint extension: ' + value)
            inventory[value] = record
    return inventory

def l6_validate_audit_cache(saved):
    discovery = saved['discovery']
    critical_errors = l6_critical_scan_errors(discovery)
    if critical_errors:
        raise ValueError('Previous source scan had critical errors: ' + repr(critical_errors))
    ignored = [e for e in discovery.get('scan_errors', []) if l6_is_noncritical_scan_error(e)]
    if ignored:
        print(f'Ignoring {len(ignored)} noncritical dataset scan error(s) from cached audit.')
    core = l6_choose_bundle(discovery['sources'], CORE_NAMES, L6_SOURCE_DIR_OVERRIDE)
    shared = l6_choose_bundle(discovery['sources'], SHARED_NAMES, L6_SHARED_DIR_OVERRIDE)
    selected = {}
    for names, resolution, stored in ((CORE_NAMES,core,saved['core_resolution']),
                                      (SHARED_NAMES,shared,saved['shared_resolution'])):
        if not resolution['selected'] or resolution['selected'] != stored['selected']:
            raise ValueError('Source selection missing, ambiguous or override changed')
        selected.update({name:Path(resolution['selected'])/name for name in names})
    print('Verifying cached source paths...')
    for name in REQUIRED_NAMES:
        record = saved['sources'][name]
        path = l6_audit_path(record['path'])
        if path != selected[name] or str(path) not in discovery['sources'][name]:
            raise ValueError('Source path/selection mismatch: ' + name)
        if l6_audit_source_hash(path) != record['sha256']:
            raise ValueError('Source SHA256 changed: ' + str(path))
        if record.get('parse_error'):
            raise ValueError('Cached source contains syntax error: ' + str(path))
        for candidate in discovery['sources'][name]:
            if not l6_audit_path(candidate).is_file():
                raise FileNotFoundError('Missing source candidate: ' + candidate)
    print('Source hashes: OK')
    if set(saved['reltr']) != set(discovery['reltr']) or set(saved['caches']) != set(discovery['caches']):
        raise ValueError('Cached discovery and audit inventories differ')
    inventory = l6_audit_inventory(discovery, saved.get('path_inventory'))
    print('RelTR paths: OK')
    print('Checkpoint/cache paths: OK')
    return selected, core, shared, inventory

def l6_audit_narrow_roots(saved=None):
    # Known paths only; no directory traversal to look for shortcuts on a cache hit.
    roots = [SEARCH_ROOT/'MyDrive/DataKhoaLuan']
    hints = [L6_SOURCE_DIR_OVERRIDE, L6_SHARED_DIR_OVERRIDE]
    if isinstance(saved, dict):
        hints += saved.get('search_roots', [])
        for record in saved.get('sources', {}).values():
            if isinstance(record, dict): hints.append(record.get('path'))
    for hint in hints:
        if not isinstance(hint, (str, os.PathLike)): continue
        path = Path(hint)
        for ancestor in (path, *path.parents):
            if ancestor.name == 'DataKhoaLuan' and ancestor.resolve().is_relative_to(SEARCH_ROOT.resolve()):
                roots.append(ancestor); break
    return list(dict.fromkeys(roots))

def l6_discover_for_audit(saved=None):
    roots = l6_audit_narrow_roots(saved)
    found = l6_scan_drive_roots(roots, LEVEL6_DIR)
    complete = all(found['sources'].get(name) for name in REQUIRED_NAMES)
    # An ambiguous complete bundle is reported, never resolved by broadening the search.
    if not complete:
        print('Source incomplete under DataKhoaLuan; falling back to broad Drive roots.')
        roots = list(DISCOVERY_ROOTS)
        found = l6_scan_drive_roots(roots, LEVEL6_DIR)
    return found, roots

AUDIT_CACHE_PATH = l6_output_path('debug/source_audit.json')
AUDIT_CACHED_PAYLOAD = None
AUDIT_REUSED = False
AUDIT_PATH_INVENTORY = {}
if AUDIT_CACHE_PATH.is_file():
    print('SOURCE AUDIT CACHE FOUND')
    try:
        AUDIT_CACHED_PAYLOAD = json.loads(AUDIT_CACHE_PATH.read_text(encoding='utf-8'))
        if not FORCE_SOURCE_RESCAN:
            SOURCE_PATHS, CORE_RESOLUTION, SHARED_RESOLUTION, AUDIT_PATH_INVENTORY = l6_validate_audit_cache(AUDIT_CACHED_PAYLOAD)
            DISCOVERY = AUDIT_CACHED_PAYLOAD['discovery']
            SOURCE_AUDIT = AUDIT_CACHED_PAYLOAD['sources']
            RELTR_AUDIT = AUDIT_CACHED_PAYLOAD['reltr']
            CACHE_AUDIT = AUDIT_CACHED_PAYLOAD['caches']
            L5_SOURCE_HASHES_BEFORE = {r['path']:r['sha256'] for r in SOURCE_AUDIT.values()}
            AUDIT_REUSED = True
            print('SOURCE AUDIT CACHE REUSED')
    except (OSError, ValueError, TypeError, KeyError, AttributeError) as exc:
        print('SOURCE AUDIT CACHE INVALID')
        print('Reason:', str(exc))
if not AUDIT_REUSED:
    print('RESCANNING...' + (' FORCE_SOURCE_RESCAN=True' if FORCE_SOURCE_RESCAN else ''))
    DISCOVERY, AUDIT_SEARCH_ROOTS = l6_discover_for_audit(AUDIT_CACHED_PAYLOAD)
    print('Missing search roots (shortcut folder can legitimately be absent):',
          DISCOVERY['missing_search_roots'])
    CORE_RESOLUTION = l6_choose_bundle(DISCOVERY['sources'], CORE_NAMES, L6_SOURCE_DIR_OVERRIDE)
    SHARED_RESOLUTION = l6_choose_bundle(DISCOVERY['sources'], SHARED_NAMES, L6_SHARED_DIR_OVERRIDE)
    # Only source-relevant scan errors invalidate source uniqueness. Dataset I/O errors are warnings.
    CRITICAL_SCAN_ERRORS = l6_critical_scan_errors(DISCOVERY)
    NONCRITICAL_SCAN_ERRORS = [e for e in DISCOVERY.get('scan_errors', [])
                               if l6_is_noncritical_scan_error(e)]
    if NONCRITICAL_SCAN_ERRORS:
        print(f'WARNING: ignored {len(NONCRITICAL_SCAN_ERRORS)} noncritical dataset scan error(s).')
        for err in NONCRITICAL_SCAN_ERRORS[:10]:
            print('  NONCRITICAL:', err)
    if CRITICAL_SCAN_ERRORS:
        for resolution in (CORE_RESOLUTION, SHARED_RESOLUTION):
            resolution.update(selected=None, reason='Critical source scan error; uniqueness not verified')
    SOURCE_PATHS = {}
    for names, resolution in ((CORE_NAMES, CORE_RESOLUTION), (SHARED_NAMES, SHARED_RESOLUTION)):
        if resolution['selected']:
            SOURCE_PATHS.update({name: Path(resolution['selected']) / name for name in names})
    SOURCE_AUDIT = {}
    for name in REQUIRED_NAMES:
        path = SOURCE_PATHS.get(name)
        record = dict(path=str(path) if path else None, selected=path is not None,
                      candidates=DISCOVERY['sources'][name])
        if path is not None:
            source_hash = l6_audit_source_hash(path)
            text = path.read_text(encoding='utf-8-sig')
            record.update(sha256=source_hash, source=text)
            try:
                tree = ast.parse(text)
                record['definitions'] = [dict(name=n.name, line=n.lineno,
                    signature=(ast.unparse(n.args) if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)) else None))
                    for n in ast.walk(tree)
                    if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef))]
                record['imports'] = [ast.unparse(n) for n in ast.walk(tree)
                                     if isinstance(n, (ast.Import, ast.ImportFrom))]
                record['checkpoint_filename_literals'] = sorted({
                    n.value for n in ast.walk(tree) if isinstance(n, ast.Constant)
                    and isinstance(n.value, str) and n.value.lower().endswith(('.pt', '.pth', '.ckpt'))})
                terms = ('DualBranchDecoderLayer', 'knowledge_proj', 'C_vis', 'C_knw',
                         'gate', 'mmr_select', 'load_state_dict', 'torch.load', 'beam', 'encode_text')
                record['integration_locations'] = [dict(line=i, text=line)
                    for i, line in enumerate(text.splitlines(), 1) if any(term in line for term in terms)]
            except SyntaxError as exc:
                record['parse_error'] = str(exc)
        SOURCE_AUDIT[name] = record
    print('SOURCE FILES FOUND')
    print(json.dumps(dict(core_resolution=CORE_RESOLUTION, shared_resolution=SHARED_RESOLUTION),
                     ensure_ascii=False, indent=2))
    for name, record in SOURCE_AUDIT.items():
        print(name, 'SELECTED:', record['path'], '| CANDIDATES:', record['candidates'])
        for location in record.get('integration_locations', []):
            print('  ', location['line'], location['text'])

    RELTR_AUDIT = {}
    for value in DISCOVERY['reltr']:
        path = Path(value)
        record = dict(path=value, schema_status='observed_not_semantically_mapped')
        try:
            with path.open('r', encoding='utf-8-sig', newline='') as f:
                sample = f.read(65536)
                f.seek(0)
                # Sniff delimiter; CSV semantics/SPO are not inferred from delimiter or column names.
                try:
                    dialect = csv.Sniffer().sniff(sample, delimiters=',;\t|')
                except csv.Error:
                    dialect = csv.excel
                    record['delimiter_warning'] = 'Sniff failed; preview uses comma, parser not approved'
                reader = csv.DictReader(f, dialect=dialect)
                record.update(columns=reader.fieldnames, sample_rows=[], delimiter=dialect.delimiter)
                for row in reader:
                    record['sample_rows'].append(row)
                    if len(record['sample_rows']) == 3:
                        break
            record['size_bytes'] = path.stat().st_size
            record['hash_status'] = 'deferred_until_actual_reuse'
        except (OSError, UnicodeError, csv.Error) as exc:
            record['read_error'] = str(exc)
        RELTR_AUDIT[value] = record
    print('RELTR FILES FOUND')
    print(json.dumps(RELTR_AUDIT, ensure_ascii=False, indent=2))
    print('CHECKPOINTS FOUND')
    print(json.dumps(DISCOVERY['checkpoints'], ensure_ascii=False, indent=2))
    print('TRAINED / SEED DIRECTORIES FOUND')
    print(json.dumps(dict(trained=DISCOVERY['trained_directories'], seeds=DISCOVERY['seed_directories']), indent=2))
    CACHE_AUDIT = {path: dict(path=path, hash_status='deferred_until_actual_reuse')
                   for path in DISCOVERY['caches']}
    print('CACHE FOUND')
    print(json.dumps(list(CACHE_AUDIT), ensure_ascii=False, indent=2))
    L5_SOURCE_HASHES_BEFORE = {r['path']: r['sha256'] for r in SOURCE_AUDIT.values() if 'sha256' in r}

    # Persist only a complete, uniquely resolved audit. /4b may still inspect discovery after a failure.
    problems = [r.get('reason') for r in (CORE_RESOLUTION, SHARED_RESOLUTION) if not r['selected']]
    problems += l6_critical_scan_errors(DISCOVERY)
    problems += [name + ': ' + r['parse_error'] for name,r in SOURCE_AUDIT.items() if r.get('parse_error')]
    if problems:
        raise RuntimeError('Source audit incomplete; reusable cache not replaced: ' + repr(problems))
    AUDIT_PATH_INVENTORY = l6_audit_inventory(DISCOVERY)
    for path,record in CACHE_AUDIT.items():
        record['size_bytes'] = AUDIT_PATH_INVENTORY[path]['size_bytes']
    AUDIT_CACHED_PAYLOAD = dict(schema_version=AUDIT_SCHEMA_VERSION, timestamp=l6_now(),
        search_root=str(SEARCH_ROOT), search_roots=[str(p) for p in AUDIT_SEARCH_ROOTS],
        discovery=DISCOVERY, core_resolution=CORE_RESOLUTION, shared_resolution=SHARED_RESOLUTION,
        sources=SOURCE_AUDIT, source_paths={name:str(path) for name,path in SOURCE_PATHS.items()},
        source_hashes=L5_SOURCE_HASHES_BEFORE, reltr=RELTR_AUDIT, caches=CACHE_AUDIT,
        path_inventory=AUDIT_PATH_INVENTORY)
    l6_write_json('debug/source_audit.json', AUDIT_CACHED_PAYLOAD)
    print('Audit saved:', AUDIT_CACHE_PATH)
elif not AUDIT_CACHED_PAYLOAD.get('path_inventory'):
    # Upgrade a valid legacy audit without any Drive scan or large-file hashing.
    AUDIT_CACHED_PAYLOAD.update(schema_version=AUDIT_SCHEMA_VERSION,
        path_inventory=AUDIT_PATH_INVENTORY,
        source_paths={name:str(path) for name,path in SOURCE_PATHS.items()},
        source_hashes=L5_SOURCE_HASHES_BEFORE)
    l6_write_json('debug/source_audit.json', AUDIT_CACHED_PAYLOAD)

def l6_assert_source_unchanged():
    for path, expected in L5_SOURCE_HASHES_BEFORE.items():
        assert l6_audit_source_hash(path) == expected, f'Source changed: {path}'
    return len(L5_SOURCE_HASHES_BEFORE)

print('Source Audit ready; /4b optional. Dataset image folders are pruned; noncritical COCO/Flickr I/O errors do not invalidate source audit.')
print('Continue /5 through /11 for the selected seed.')


Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
  "/content/drive/.shortcut-targets-by-id/1OCkkcQKlIYTCPLnGgb2y4aTf_VJFE7xQ/Data/baseline_level5/lv5_A/trained_5tokens_v1/cache/07d76afb905955b707ba (188)/w0.6_lambda0.6/train/094000.npz",
  "/content/drive/.shortcut-targets-by-id/1OCkkcQKlIYTCPLnGgb2y4aTf_VJFE7xQ/Data/baseline_level5/lv5_A/trained_5tokens_v1/cache/07d76afb905955b707ba (188)/w0.6_lambda0.6/train/095000.npz",
  "/content/drive/.shortcut-targets-by-id/1OCkkcQKlIYTCPLnGgb2y4aTf_VJFE7xQ/Data/baseline_level5/lv5_A/trained_5tokens_v1/cache/07d76afb905955b707ba (189)/w0.6_lambda0.6/train/092000.npz",
  "/content/drive/.shortcut-targets-by-id/1OCkkcQKlIYTCPLnGgb2y4aTf_VJFE7xQ/Data/baseline_level5/lv5_A/trained_5tokens_v1/cache/07d76afb905955b707ba (189)/w0.6_lambda0.6/train/093000.npz",
  "/content/drive/.shortcut-targets-by-id/1OCkkcQKlIYTCPLnGgb2y4aTf_VJFE7xQ/Data/baseline_level5/lv5_A/trained_5tokens_v1/cache/07d76afb905955b707ba (190)/w0.6_lambda0.6/train/090000.npz"

## Level 6 / 4b — Gom source Level 5A để mang về Codex kiểm tra

Chạy ngay sau **Level 6 / 4 — Source Audit**. Cell chỉ đọc source, phân tích import bằng AST
và ghi `debug/level5a_source_bundle.zip` trong LEVEL6_DIR. Không execute/import module Level 5A.
ZIP giữ đường dẫn tương đối dưới `sources/`, kèm `source_manifest.json` và `reltr_sample.csv`
(header thật + tối đa 20 dòng thật). Không đóng gói checkpoint, embeddings hoặc dataset.

Nếu trùng phiên bản chưa được chọn, các candidate được đóng gói và đánh dấu để phân tích.
Import động hoặc module không resolve được được ghi trong manifest; không tạo source giả.
Giới hạn bảo vệ: 2 MiB/file Python, 32 MiB tổng source và 500 file; phần bị loại được báo rõ.
**Cell /4b là tùy chọn để xuất source bundle. Để inference, tiếp tục /5 → /11.**


In [ ]:
# Gom source Level 5A để mang về Codex kiểm tra (không import/execute source).
import io, itertools, tokenize, zipfile

RELTR_SAMPLE_SOURCE = None  # Có thể đặt path CSV cụ thể từ RELTR_AUDIT; mặc định ưu tiên val.
MAX_SOURCE_FILE_BYTES = 2 * 1024 * 1024
MAX_BUNDLE_SOURCE_BYTES = 32 * 1024 * 1024
MAX_SOURCE_FILES = 500


def l6_build_source_bundle():
    required_globals = ('DISCOVERY', 'SOURCE_AUDIT', 'SOURCE_PATHS', 'RELTR_AUDIT',
                        'SEARCH_ROOT', 'LEVEL6_DIR', 'REQUIRED_NAMES')
    absent = [name for name in required_globals if name not in globals()]
    if absent:
        raise RuntimeError('Chạy Level 6 / 1–4 trước: thiếu ' + ', '.join(absent))
    search_root = SEARCH_ROOT.resolve()
    output_root = LEVEL6_DIR.resolve()
    bundle_path = l6_output_path('debug/level5a_source_bundle.zip')
    missing, warnings, unresolved, import_resolutions = [], [], [], []
    initial = []
    # Khi Source Audit chưa chọn được vì trùng bản, gom các candidate kèm nhãn rõ ràng.
    # Không tự chọn phiên bản làm nguồn inference.
    for name in REQUIRED_NAMES:
        chosen = SOURCE_PATHS.get(name)
        candidates = [str(chosen)] if chosen else DISCOVERY['sources'].get(name, [])
        if not candidates:
            missing.append(name)
        if not chosen and candidates:
            warnings.append(f'{name}: chưa chọn phiên bản; ZIP chứa các candidate để phân tích')
        initial.extend(Path(value).resolve() for value in candidates)
    initial = sorted(set(initial), key=str)

    def source_tree(path):
        # Ưu tiên biên source tree baseline_level5, không quét toàn bộ DataKhoaLuan.
        for parent in path.parents:
            if parent.name.casefold() == 'baseline_level5':
                return parent
        return path.parent

    trees = sorted(set(source_tree(path) for path in initial), key=str)
    import_roots = sorted(set(path.parent for path in initial), key=str)
    import_roots += [tree for tree in trees if tree not in import_roots]

    def allowed(path):
        path = path.resolve()
        return (path.suffix == '.py' and path.is_relative_to(search_root)
                and not path.is_relative_to(output_root)
                and any(path.is_relative_to(tree) for tree in trees))

    def module_files(base, parts):
        if not parts or any(not part.isidentifier() for part in parts):
            return []
        target = base.joinpath(*parts)
        # Có cả module và package cùng tên thì giữ cả hai để phân tích, không execute.
        result = [candidate.resolve() for candidate in
                  (target.with_suffix('.py'), target / '__init__.py')
                  if candidate.is_file() and allowed(candidate)]
        return result

    def package_initializers(path):
        result = []
        for parent in path.parents:
            if not parent.is_relative_to(search_root):
                break
            candidate = parent / '__init__.py'
            if candidate.is_file() and allowed(candidate):
                result.append(candidate.resolve())
        return result

    queue = list(initial)
    visited, payloads, records = set(), {}, []
    total_bytes = 0
    while queue:
        path = queue.pop(0).resolve()
        if path in visited:
            continue
        visited.add(path)
        if not allowed(path) or not path.is_file():
            missing.append(f'Source không có hoặc ngoài source tree: {path}')
            continue
        if len(records) >= MAX_SOURCE_FILES:
            missing.append(f'Giới hạn {MAX_SOURCE_FILES} source files; chưa gom: {path}')
            continue
        try:
            if path.stat().st_size > MAX_SOURCE_FILE_BYTES:
                missing.append(f'Python source vượt giới hạn 2 MiB: {path}')
                continue
            with path.open('rb') as handle:
                data = handle.read(MAX_SOURCE_FILE_BYTES + 1)
            if len(data) > MAX_SOURCE_FILE_BYTES or total_bytes + len(data) > MAX_BUNDLE_SOURCE_BYTES:
                missing.append(f'Vượt giới hạn kích thước source bundle: {path}')
                continue
            digest = hashlib.sha256(data).hexdigest()
            original_digest = SOURCE_AUDIT.get(path.name, {}).get('sha256')
            original_path = SOURCE_AUDIT.get(path.name, {}).get('path')
            if original_path == str(path) and original_digest and original_digest != digest:
                raise RuntimeError(f'Source thay đổi từ lần Source Audit; chạy lại audit: {path}')
            arcname = 'sources/' + path.relative_to(search_root).as_posix()
            payloads[arcname] = data
            total_bytes += len(data)
            records.append(dict(original_path=str(path), file_name=path.name,
                source_root=str(source_tree(path)), archive_path=arcname,
                size_bytes=len(data), sha256=digest,
                selected_by_source_audit=(SOURCE_PATHS.get(path.name) is not None
                    and Path(SOURCE_PATHS[path.name]).resolve() == path)))
            queue.extend(package_initializers(path))
            encoding, _ = tokenize.detect_encoding(io.BytesIO(data).readline)
            tree = ast.parse(data.decode(encoding), filename=str(path))
        except (OSError, UnicodeError, SyntaxError) as exc:
            warnings.append(f'Không đọc/parse được {path}: {exc}')
            if not any(r['original_path'] == str(path) for r in records):
                missing.append(str(path))
            continue
        # Chỉ phân tích AST; tuyệt đối không import hoặc chạy code Level5A.
        bases = list(import_roots)
        for parent in path.parents:
            if any(parent.is_relative_to(root) for root in trees) and parent not in bases:
                bases.append(parent)
        for node in ast.walk(tree):
            requests = []
            if isinstance(node, ast.Import):
                requests = [(alias.name, 0) for alias in node.names]
            elif isinstance(node, ast.ImportFrom):
                if node.module:
                    requests.append((node.module, node.level))
                requests += [('.'.join(filter(None, [node.module, alias.name])), node.level)
                             for alias in node.names if alias.name != '*']
            elif isinstance(node, ast.Call):
                function = ast.unparse(node.func)
                if function in ('__import__', 'importlib.import_module', 'import_module'):
                    if node.args and isinstance(node.args[0], ast.Constant) and isinstance(node.args[0].value, str):
                        name = node.args[0].value
                        requests = [(name.lstrip('.'), len(name) - len(name.lstrip('.')))]
                    else:
                        unresolved.append(dict(source=str(path), line=node.lineno,
                            kind='dynamic_import_requires_review', expression=ast.unparse(node)))
            for module, level in requests:
                parts = module.split('.') if module else []
                if level:
                    base = path.parent
                    for _ in range(level - 1):
                        base = base.parent
                    search_bases = [base]
                else:
                    search_bases = bases
                matches = set()
                for base in search_bases:
                    # Include package prefixes as well as leaf modules.
                    for count in range(1, len(parts) + 1):
                        matches.update(module_files(base, parts[:count]))
                if matches:
                    import_resolutions.append(dict(source=str(path), line=node.lineno,
                        module='.' * level + module, candidates=sorted(map(str, matches)),
                        note='static source collection only; not runtime import resolution'))
                    queue.extend(sorted(matches, key=str))
                else:
                    unresolved.append(dict(source=str(path), line=node.lineno,
                        module='.' * level + module, kind='external_or_unresolved_internal_import'))

    # CSV sample luôn nằm trong ZIP; không copy CSV gốc vào output.
    reltr_info = dict(original_path=None, archive_path=None, row_count=0)
    csv_candidates = sorted(RELTR_AUDIT, key=lambda value: (
        0 if 'val' in Path(value).stem.casefold() else
        1 if 'test' in Path(value).stem.casefold() else 2, value))
    if RELTR_SAMPLE_SOURCE is not None:
        requested = str(Path(RELTR_SAMPLE_SOURCE).resolve())
        if requested not in RELTR_AUDIT:
            raise ValueError('RELTR_SAMPLE_SOURCE phải là CSV đã được Source Audit tìm thấy')
        csv_candidates = [requested]
    for value in csv_candidates:
        audit = RELTR_AUDIT[value]
        if audit.get('read_error') or audit.get('delimiter_warning'):
            warnings.append(f'Bỏ CSV chưa xác minh delimiter/readability: {value}')
            continue
        csv_path = Path(value).resolve()
        if not csv_path.is_relative_to(search_root) or csv_path.is_relative_to(output_root):
            continue
        try:
            with csv_path.open('r', encoding='utf-8-sig', newline='') as handle:
                # Sniff lại đầy đủ dialect để giữ quoting, không chỉ delimiter.
                prefix = handle.read(65536)
                handle.seek(0)
                dialect = csv.Sniffer().sniff(prefix, delimiters=',;\t|')
                reader = csv.reader(handle, dialect=dialect)
                header = next(reader, None)
                if not header:
                    warnings.append(f'CSV không có header: {value}')
                    continue
                rows = list(itertools.islice(reader, 20))
            buffer = io.StringIO(newline='')
            writer = csv.writer(buffer, dialect=dialect)
            writer.writerow(header)
            writer.writerows(rows)
            sample = buffer.getvalue().encode('utf-8')
            payloads['reltr_sample.csv'] = sample
            reltr_info = dict(original_path=value, archive_path='reltr_sample.csv',
                header=header, row_count=len(rows), delimiter=dialect.delimiter,
                sample_sha256=hashlib.sha256(sample).hexdigest(),
                selection_reason='explicit override' if RELTR_SAMPLE_SOURCE else 'deterministic val/test/other preference',
                original_file_included=False)
            break
        except (OSError, UnicodeError, csv.Error) as exc:
            warnings.append(f'Không lấy mẫu được {value}: {exc}')
    if reltr_info['archive_path'] is None:
        missing.append('reltr_sample.csv: chưa có RelTR CSV đọc được; không tạo mẫu giả')

    # Cell này chạy trước checkpoint resolver, nên mapping có thể chưa được xác minh.
    # Không yêu cầu chạy cell loader/smoke chỉ để lấy source bundle.
    checkpoint_mapping = {}
    for seed in SEEDS:
        candidates = [value for value in DISCOVERY['checkpoints']
                      if f'seed{seed}' in Path(value).parts and l6_priority(value) == 0]
        checkpoint_mapping[str(seed)] = dict(candidates=candidates,
            selected_path=None, load_verified=False,
            status='candidate_inventory_only; loader/best-role not verified')
    manifest = dict(purpose='source_analysis_only_not_dataset_backup', timestamp=l6_now(),
        search_root=str(search_root), bundle_path=str(bundle_path),
        source_roots=[str(root) for root in trees], sources=records,
        required_source_names=list(REQUIRED_NAMES),
        source_audit_selection={name: str(path) for name, path in SOURCE_PATHS.items()},
        reltr=reltr_info, reltr_original_path=reltr_info['original_path'],
        checkpoint_candidates=DISCOVERY['checkpoints'],
        cache_paths=DISCOVERY['caches'], seed_checkpoint_mapping=checkpoint_mapping,
        prior_checkpoint_resolution=globals().get('CHECKPOINT_RESOLUTION'),
        prior_checkpoint_resolution_note='If present, from an earlier cell execution; not revalidated by bundling.',
        import_resolutions=import_resolutions, unresolved_imports=unresolved,
        missing=missing, warnings=warnings, scan_errors=DISCOVERY.get('scan_errors', []),
        excluded_large_files=['checkpoints', 'embeddings', 'image features', 'datasets', 'full RelTR CSV'],
        limitations=['Static import closure; dynamic sys.path/imports require source review.',
                    'Unresolved imports may be standard-library/third-party or missing internal modules.'],
        complete_requested_sources=all(any(r['file_name'] == name for r in records) for name in REQUIRED_NAMES))
    payloads['source_manifest.json'] = json.dumps(manifest, ensure_ascii=False, indent=2).encode('utf-8')
    # Build/verify ZIP in memory; filesystem write only to LEVEL6_DIR.
    archive = io.BytesIO()
    with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        for name, data in sorted(payloads.items()):
            assert name.endswith('.py') or name in ('source_manifest.json', 'reltr_sample.csv')
            z.writestr(name, data)
    archive.seek(0)
    with zipfile.ZipFile(archive) as z:
        assert z.testzip() is None
        assert set(z.namelist()) == set(payloads)
    # Re-read hashes to detect source changes during collection; never write those files.
    for record in records:
        if l6_sha256(record['original_path']) != record['sha256']:
            raise RuntimeError('Source thay đổi khi gom ZIP: ' + record['original_path'])
    bundle_path.parent.mkdir(parents=True, exist_ok=True)
    with bundle_path.open('wb') as handle:
        handle.write(archive.getvalue())
    print('LEVEL 5A SOURCE BUNDLE READY')
    print('Bundle:\n' + str(bundle_path))
    print('Included Python files:')
    for record in records:
        print(' -', record['archive_path'], '<-', record['original_path'])
    print('RelTR sample:', reltr_info['archive_path'], '| rows:', reltr_info['row_count'],
          '| original:', reltr_info['original_path'])
    print('Excluded large files:\ncheckpoints / embeddings / image features / datasets')
    print('MISSING:')
    for item in missing or ['None']:
        print(' -', item)
    if warnings:
        print('WARNINGS:')
        for item in warnings:
            print(' -', item)
    print('Import không resolve được:', len(unresolved), '(chi tiết trong source_manifest.json)')
    print('Tải ZIP từ panel Files của Colab hoặc chạy files.download(str(L6_SOURCE_BUNDLE_PATH)).')
    return bundle_path

L6_SOURCE_BUNDLE_PATH = l6_build_source_bundle()
# Tuỳ chọn tải trực tiếp sau khi ZIP sẵn sàng:
# from google.colab import files
# files.download(str(L6_SOURCE_BUNDLE_PATH))

## Level 6 /5 — Source-backed retrieval, Top-50 reuse/fallback và RelTR parser

Caption retrieval kế thừa pipeline Level 5A: Min-Max normalization, semantic expansion và MMR.

Số caption được chọn là **động từ 2 đến 5**, tùy số relation RelTR thật có sẵn cho ảnh đó, để tổng luôn đúng 5 knowledge tokens.

Top-50 caption candidates:
- ưu tiên reuse cache Level 5A tương thích theo chế độ READ-ONLY;
- nếu không có, reuse cache Level 6;
- nếu vẫn thiếu, tự tính lại Top-50 bằng FAISS từ COCO_2014 image features và caption embeddings đã có;
- cache mới chỉ được ghi vào `baseline_level6/_Chi_Dung_TriThuc/cache`, không ghi vào Level 5A.

RelTR schema được dùng là `image,prediction,score`.
`l6_parse_reltr` tách `prediction` thành subject/predicate/object, dedup canonical SPO và giữ confidence tốt nhất.

Relation selection:
- confidence RelTR;
- CLIP/image-feature similarity;
- Min-Max normalization;
- MMR;
- chọn tối đa 3 SPO riêng biệt.


In [12]:
REVIEWED_SOURCE_HASHES = {
 'level5a_pipeline.py':'5e73694a07427e8e251d1f183c96aea94a247c3d8e19f44e61933701dd4df086',
 'level5_model.py':'004e171a7349de30ee8cd453af680cb0dace33420704e13cefcd9d3e5ba2ae05',
 'coco_eval_worker.py':'7d8cf752e85fbab35a6ff275c514990fa81be0d777c895312625df448d39fab0',
 'multiseed.py':'17d8244b2922f8596d646d3472f5045e78f3b589df7418f7ba923162006e0497',
 'storage.py':'cad0c9d25491f98e04c27de5ed0e1199bc99b7581e6e134e2fcff831684508b1',
 'semantic_expansion.py':'126119e4484b8d0419b853f1752321e59a44f8f65544b74d838368ef4f8eb9fe',
 'mmr_diversity.py':'a3f6944129aefc33ad3e57350c93ad9ddd873e0546eaf0575d42dfeea44cf805'}

def l6_setup_modules():
    problems = []
    for name, expected in REVIEWED_SOURCE_HASHES.items():
        path = SOURCE_PATHS.get(name)
        if path is None or not path.is_file():
            problems.append(f'MISSING/AMBIGUOUS {name}: {DISCOVERY["sources"].get(name, [])}')
        elif l6_sha256(path) != expected:
            problems.append(f'UNREVIEWED SOURCE VERSION: {path}; need source review before applying adapter')
    if problems:
        raise RuntimeError('\n'.join(problems))
    critical_scan_errors = l6_critical_scan_errors(DISCOVERY)
    if critical_scan_errors:
        raise RuntimeError('Source scan incomplete (critical only): ' + repr(critical_scan_errors))
    ignored_scan_errors = [e for e in DISCOVERY.get('scan_errors', []) if l6_is_noncritical_scan_error(e)]
    if ignored_scan_errors:
        print(f'Ignoring {len(ignored_scan_errors)} noncritical dataset scan error(s) during module setup.')
    # Hard guard for Level5 filesystem writes, including accidental recovery calls.
    if not getattr(sys, '_level6_readonly_guard', False):
        def guard(event, args):
            paths = []
            if event == 'open':
                name, mode, flags = args
                write = (isinstance(mode, str) and any(c in mode for c in 'wax+')) or (
                    isinstance(flags, int) and flags & (os.O_WRONLY | os.O_RDWR | os.O_CREAT | os.O_TRUNC))
                if write: paths = [name]
            elif event in ('os.remove', 'os.rmdir', 'os.mkdir', 'os.chmod', 'os.utime', 'os.truncate'):
                paths = args[:1]
            elif event in ('os.rename', 'os.link', 'os.symlink'):
                paths = args[:2]
            for raw in paths:
                if isinstance(raw, (str, bytes, os.PathLike)):
                    path = Path(os.fsdecode(raw)).resolve()
                    if 'baseline_level5' in [v.lower() for v in path.parts]:
                        raise PermissionError('Level5 is READ ONLY: ' + str(path))
        sys.addaudithook(guard)
        sys._level6_readonly_guard = True
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'nltk==3.9.1', 'pycocoevalcap==1.2', 'faiss-cpu'], check=True)
    import nltk
    assert nltk.__version__ == '3.9.1', 'Restart Colab: different NLTK already imported'
    nltk_dir = l6_output_path('cache/nltk_data')
    nltk.data.path.insert(0, str(nltk_dir))
    for corpus in ('wordnet', 'omw-1.4'):
        nltk.download(corpus, download_dir=str(nltk_dir), quiet=True, raise_on_error=True)
    code_dir = SOURCE_PATHS['level5_model.py'].parent.resolve()
    shared_dir = SOURCE_PATHS['semantic_expansion.py'].parent.resolve()
    for directory in (shared_dir, code_dir):
        if str(directory) in sys.path: sys.path.remove(str(directory))
        sys.path.insert(0, str(directory))
    for name in ('multiseed','storage','level5a_pipeline','level5_model','semantic_expansion','mmr_diversity'):
        sys.modules.pop(name, None)
    M = importlib.import_module('level5_model')
    S = importlib.import_module('storage')
    P = importlib.import_module('level5a_pipeline')
    for name in ('level5_model','storage','level5a_pipeline','semantic_expansion','mmr_diversity'):
        assert Path(sys.modules[name].__file__).resolve() == SOURCE_PATHS[name + '.py'].resolve()
    source = textwrap.dedent(inspect.getsource(P.rerank_one))
    for old in ('def rerank_one(', 'k=5', 'len(selected) != 5'):
        assert source.count(old) == 1, 'rerank_one implementation changed'
    rerankers = {}
    for k in (2, 3, 4, 5):
        adapted = source.replace('def rerank_one(', f'def l6_rerank_k{k}(').replace('k=5', f'k={k}')
        adapted = adapted.replace('len(selected) != 5', f'len(selected) != {k}').replace('Không đủ 5', f'Không đủ {k}')
        namespace = dict(vars(P))
        exec(compile(adapted, f'<Level6-rerank-k{k}>', 'exec'), namespace)
        rerankers[k] = namespace[f'l6_rerank_k{k}']
    return dict(P=P, M=M, S=S, rerankers=rerankers, code_dir=code_dir)

def l6_parse_reltr(path, allowed_ids):
    relations, labels, seen_ids = {}, {}, set()
    with Path(path).open(encoding='utf-8-sig', newline='') as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames != ['image', 'prediction', 'score']:
            raise ValueError(f'Unsupported RelTR schema at {path}: {reader.fieldnames}; expected reviewed schema')
        for line_no, row in enumerate(reader, 2):
            match = re.search(r'_(\d+)\.jpg$', row['image'])
            if not match: raise ValueError(f'Invalid image at {path}:{line_no}')
            iid = int(match.group(1)); seen_ids.add(iid)
            score = float(row['score'])
            if not math.isfinite(score): raise ValueError(f'Non-finite RelTR confidence: line {line_no}')
            relations.setdefault(iid, {}); labels.setdefault(iid, set())
            if row['prediction'].strip() == 'no_relation': continue
            parts = [x.strip() for x in row['prediction'].split(' - ', 2)]
            if len(parts) != 3 or not all(parts): raise ValueError(f'Invalid SPO: line {line_no}')
            subject, predicate, obj = parts
            key = tuple(' '.join(v.replace('_',' ').lower().split()) for v in parts)
            text = ' '.join(' '.join(v.replace('_',' ').split()) for v in parts)
            item = dict(type='relation', subject=subject, predicate=predicate, object=obj,
                        serialized_text=text, confidence=score, source=str(path), source_line=line_no)
            prior = relations[iid].get(key)
            if prior is None or score > prior['confidence']:
                relations[iid][key] = item
            labels[iid].update((subject.lower(), obj.lower()))
    if seen_ids != set(allowed_ids):
        raise ValueError('RelTR IDs differ from original split; missing=' + repr(sorted(set(allowed_ids)-seen_ids)[:10]))
    return ({iid:[items[k] for k in sorted(items)] for iid,items in relations.items()},
            {iid:sorted(items) for iid,items in labels.items()})

def l6_read_cache_ro(path, expected, S):
    problems = []
    for candidate in (Path(path), Path(str(path) + '.pending')):
        if not candidate.is_file(): continue
        try:
            with np.load(candidate, allow_pickle=False) as archive:
                meta = json.loads(str(archive['metadata_json'].item()))
                arrays = {key:archive[key] for key in archive.files if key != 'metadata_json'}
            if meta.get('cache_sha256') != S.payload_hash(arrays, meta):
                raise ValueError('Cache content hash mismatch')
            if any(meta.get(key) != value for key,value in expected.items()):
                raise ValueError('Cache belongs to another contract')
            return arrays, meta, candidate
        except (OSError, ValueError, KeyError, EOFError, zipfile.BadZipFile) as exc:
            problems.append(str(exc))
    raise RuntimeError(f'No compatible READ-ONLY cache: {path}; {problems}')


## Level 6 /6 — Knowledge memory động, projection 512→256

Kho tri thức luôn chứa đúng **5 item**.

Thành phần phụ thuộc số relation RelTR thật:

- `2 caption + 3 relation`
- `3 caption + 2 relation`
- `4 caption + 1 relation`
- `5 caption`

Caption luôn đứng trước relation trong knowledge memory.

Mỗi knowledge item được biểu diễn bằng cùng encoder 512d, sau đó đi qua `knowledge_proj` gốc:

`[1,5,512] → [1,5,256]`

Không thay đổi kiến trúc checkpoint Level 5A.


In [13]:
def l6_clean_text(value):
    return ' '.join(unicodedata.normalize('NFKC', value).split())

def l6_load_text_encoder(metadata_path, device):
    metadata = l6_read_json_ro(metadata_path)
    assert metadata['normalization'] == 'L2'
    assert metadata['caption_cleaning']['version'] == 'nfkc_whitespace_v1'
    version = metadata['transformers_version']
    try: current = importlib.metadata.version('transformers')
    except importlib.metadata.PackageNotFoundError: current = None
    if current != version:
        if 'transformers' in sys.modules:
            raise RuntimeError(f'Restart Colab before loading transformers=={version}')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==' + version], check=True)
    from transformers import AutoTokenizer, CLIPModel
    options = dict(revision=metadata['clip_model_revision'],
                   cache_dir=str(l6_output_path('cache/huggingface')), trust_remote_code=False)
    tokenizer = AutoTokenizer.from_pretrained(metadata['clip_processor_name_or_path'], **options)
    encoder = CLIPModel.from_pretrained(metadata['model_name'], **options).to(device).eval()
    encoder.requires_grad_(False)
    assert encoder.config.projection_dim == metadata['embedding_dim']
    return encoder, tokenizer, metadata

@torch.no_grad()
def l6_encode_texts(texts, encoder, tokenizer, device):
    encoded = tokenizer([l6_clean_text(text) for text in texts], padding=True, truncation=True,
        max_length=encoder.config.text_config.max_position_embeddings, return_tensors='pt')
    encoded = {key:value.to(device) for key,value in encoded.items()}
    raw = encoder.get_text_features(**encoded)
    # transformers 4 returns Tensor; reviewed transformers 5 returns projected pooler_output.
    vectors = raw if torch.is_tensor(raw) else raw.pooler_output
    vectors = torch.nn.functional.normalize(vectors.float(), p=2, dim=-1)
    assert vectors.ndim == 2 and torch.isfinite(vectors).all()
    return vectors.cpu().numpy()

def l6_relation_selection(pool, vectors, image_feature, P, lam, k):
    k = int(k)
    if k < 0 or k > 3:
        raise ValueError(f'Invalid relation k={k}')
    if k == 0:
        return [], np.empty((0, 512), dtype=np.float32), np.empty((0,), dtype=np.float32), np.empty((0,), dtype=np.float32)
    assert len(pool) == len(vectors) and len(pool) >= k
    keys = [tuple(' '.join(r[field].replace('_', ' ').lower().split())
                  for field in ('subject', 'predicate', 'object')) for r in pool]
    assert all(all(key) for key in keys), 'Incomplete SPO in relation pool'
    assert len(set(keys)) == len(keys), 'Relation pool must be deduplicated before ranking'
    query = np.asarray(image_feature, dtype=np.float64)
    norm = np.linalg.norm(query)
    if not np.isfinite(norm) or norm <= 0:
        raise ValueError('Invalid image feature')
    cosine = np.asarray(vectors, dtype=np.float64) @ (query / norm)
    confidence = np.asarray([item['confidence'] for item in pool])
    relevance = P.normalize_scores_per_image(
        0.5 * P.normalize_scores_per_image(confidence)
        + 0.5 * P.normalize_scores_per_image(cosine)
    )
    candidates = [
        dict(position=i, relevance=float(relevance[i]), embedding=vectors[i])
        for i in range(len(pool))
    ]
    chosen = P.mmr_select(candidates, k=k, lambda_param=lam)
    order = sorted((c['position'] for c in chosen), key=lambda i: (-relevance[i], i))
    assert len(order) == len(set(order)) == k
    selected = [
        dict(pool[i], score=float(relevance[i]),
             semantic_cosine=float(cosine[i]), clip_similarity=float(cosine[i]))
        for i in order
    ]
    return (selected, np.asarray(vectors[order], dtype=np.float32),
            relevance[order].astype(np.float32), relevance)

@torch.no_grad()
def l6_build_memory(caption_embeddings, relation_embeddings, model):
    dim = model.knowledge_proj.in_features
    caption_embeddings = np.asarray(caption_embeddings, dtype=np.float32)
    relation_embeddings = np.asarray(relation_embeddings, dtype=np.float32)
    if caption_embeddings.ndim != 2 or caption_embeddings.shape[1] != dim:
        raise ValueError(f'Invalid caption embedding shape: {caption_embeddings.shape}')
    if relation_embeddings.ndim != 2 or relation_embeddings.shape[1] != dim:
        raise ValueError(f'Invalid relation embedding shape: {relation_embeddings.shape}')
    if len(relation_embeddings) > 3:
        raise ValueError('At most 3 real relations are allowed')
    if len(caption_embeddings) + len(relation_embeddings) != 5:
        raise ValueError('Knowledge memory must contain exactly 5 tokens')
    k = torch.from_numpy(
        np.concatenate((caption_embeddings, relation_embeddings), axis=0)
    ).to(
        device=next(model.parameters()).device,
        dtype=model.knowledge_proj.weight.dtype
    ).unsqueeze(0)
    memory = model.build_knowledge_memory(k)
    assert k.shape == (1, 5, dim)
    assert memory.shape == (1, 5, model.knowledge_proj.out_features)
    assert torch.isfinite(k).all() and torch.isfinite(memory).all()
    return k, memory


## Level 6 /6A — Materialize và validate kho tri thức Level 6

Cell này tạo **kho tri thức Level 6 vật lý** cho toàn bộ split hiện tại.

Mỗi ảnh luôn có đúng 5 knowledge items và dùng tối đa 3 relation RelTR thật:

- >=3 relation → 2 caption + 3 relation
- 2 relation → 3 caption + 2 relation
- 1 relation → 4 caption + 1 relation
- 0 relation → 5 caption

Không duplicate relation, không fabricate relation và không dùng YOLO.

Kho được lưu dưới:

`DataKhoaLuan/Data/baseline_level6/_Chi_Dung_TriThuc/knowledge_base_level6/seedXX/`

Kho được lưu theo seed vì caption/relation ranking sử dụng `final_config` của checkpoint seed tương ứng.
Nếu manifest/hash còn hợp lệ thì notebook reuse kho đã tạo thay vì build lại.


In [14]:
def l6_kb_paths(seed, split):
    folder = l6_output_path(f'knowledge_base_level6/seed{seed}')
    folder.mkdir(parents=True, exist_ok=True)
    stem = f'{split}_max3relation_caption_fallback'
    return folder / f'{stem}.jsonl', folder / f'{stem}_manifest.json'

def l6_normalized_spo(item):
    return tuple(' '.join(str(item[field]).replace('_', ' ').lower().split())
                 for field in ('subject', 'predicate', 'object'))

def l6_validate_kb_record(record, expected_image_id=None):
    if not isinstance(record, dict):
        raise ValueError('Knowledge-base record must be a dict')
    iid = int(record['image_id'])
    if expected_image_id is not None and iid != int(expected_image_id):
        raise ValueError(f'Knowledge-base image_id mismatch: {iid} != {expected_image_id}')

    captions = record.get('captions')
    relations = record.get('relations')
    items = record.get('knowledge_items')
    types_ = record.get('knowledge_types')

    if not isinstance(captions, list) or not isinstance(relations, list):
        raise ValueError(f'image_id={iid}: captions/relations must be lists')
    relation_count = len(relations)
    caption_count = len(captions)
    if relation_count < 0 or relation_count > 3:
        raise ValueError(f'image_id={iid}: relation count must be 0..3, got {relation_count}')
    if caption_count != 5 - relation_count:
        raise ValueError(
            f'image_id={iid}: fallback must keep 5 tokens; '
            f'captions={caption_count}, relations={relation_count}'
        )
    expected_types = ['caption'] * caption_count + ['relation'] * relation_count
    if types_ != expected_types:
        raise ValueError(f'image_id={iid}: invalid knowledge_types={types_!r}')
    if not isinstance(items, list) or len(items) != 5:
        raise ValueError(f'image_id={iid}: knowledge_items must contain exactly 5 strings')

    for cap in captions:
        if cap.get('type') != 'caption' or not str(cap.get('text', '')).strip():
            raise ValueError(f'image_id={iid}: invalid caption record')
        if int(cap.get('kb_row', -1)) < 0:
            raise ValueError(f'image_id={iid}: invalid caption kb_row')
        if not math.isfinite(float(cap.get('score'))):
            raise ValueError(f'image_id={iid}: non-finite caption score')

    spo = []
    for rel in relations:
        if rel.get('type') != 'relation':
            raise ValueError(f'image_id={iid}: invalid relation type')
        key = l6_normalized_spo(rel)
        if not all(key):
            raise ValueError(f'image_id={iid}: incomplete SPO relation')
        if not str(rel.get('serialized_text', '')).strip():
            raise ValueError(f'image_id={iid}: missing serialized relation text')
        if not math.isfinite(float(rel.get('score'))):
            raise ValueError(f'image_id={iid}: non-finite relation score')
        spo.append(key)
    if len(set(spo)) != relation_count:
        raise ValueError(f'image_id={iid}: relation duplication after SPO normalization')

    expected_items = ([str(x['text']).strip() for x in captions] +
                      [str(x['serialized_text']).strip() for x in relations])
    if items != expected_items:
        raise ValueError(f'image_id={iid}: knowledge_items order/content mismatch')

    if record.get('relation_count') != relation_count or record.get('caption_count') != caption_count:
        raise ValueError(f'image_id={iid}: count metadata mismatch')
    if record.get('fallback_used') != (relation_count < 3):
        raise ValueError(f'image_id={iid}: fallback_used metadata mismatch')
    return True

def l6_select_kb_record(ctx, position):
    P = ctx['modules']['P']
    iid = int(ctx['ids'][position])
    cfg = ctx['final_config']

    pool = ctx['relations'][iid]
    relation_count = min(3, len(pool))
    caption_count = 5 - relation_count

    # Select up to 3 REAL RelTR relations. Never duplicate/fabricate.
    if relation_count:
        rel_vectors = ctx['relation_vectors'][[
            ctx['text_index'][r['serialized_text']] for r in pool
        ]]
        feature = np.asarray(ctx['features'][ctx['fmap'][iid]], dtype=np.float32)
        selected_relations, _, _, _ = l6_relation_selection(
            pool, rel_vectors, feature, P, cfg['lambda_param'], k=relation_count
        )
    else:
        selected_relations = []

    # Fill any missing relation slots with caption retrieval so total stays exactly 5.
    rows = ctx['top_rows'][position]
    emb = np.asarray(ctx['kb'][rows], dtype=np.float32)
    if not np.isfinite(emb).all():
        raise ValueError(f'image_id={iid}: non-finite caption embeddings')
    captions = [ctx['kb_captions'][int(row)] for row in rows]
    reranker = ctx['modules']['rerankers'][caption_count]
    order, cap_scores, _, fallback = reranker(
        ctx['labels'][iid], captions, emb, ctx['top_scores'][position],
        cfg['w'], cfg['lambda_param']
    )
    selected_captions = [
        dict(type='caption', text=captions[int(i)], kb_row=int(rows[i]),
             score=float(cap_scores[j]))
        for j, i in enumerate(order)
    ]

    knowledge_types = ['caption'] * caption_count + ['relation'] * relation_count
    record = dict(
        image_id=iid,
        captions=selected_captions,
        relations=selected_relations,
        knowledge_items=([x['text'] for x in selected_captions] +
                         [x['serialized_text'] for x in selected_relations]),
        knowledge_types=knowledge_types,
        caption_count=caption_count,
        relation_count=relation_count,
        fallback_used=(relation_count < 3),
        fallback_policy='caption_fill_no_yolo_no_duplicate_no_fabrication',
        caption_label_fallback=bool(fallback),
    )
    l6_validate_kb_record(record, iid)
    return record

def l6_read_level6_kb(kb_path, expected_ids):
    records = []
    with Path(kb_path).open(encoding='utf-8') as handle:
        for line_no, line in enumerate(handle, 1):
            line = line.strip()
            if not line:
                raise ValueError(f'Blank line in Level6 KB at line {line_no}')
            row = json.loads(line)
            expected = expected_ids[len(records)] if len(records) < len(expected_ids) else None
            l6_validate_kb_record(row, expected)
            records.append(row)
    if len(records) != len(expected_ids):
        raise ValueError(f'Level6 KB count mismatch: {len(records)} != {len(expected_ids)}')
    if [int(r['image_id']) for r in records] != [int(i) for i in expected_ids]:
        raise ValueError('Level6 KB image order/coverage differs from evaluation split')
    return records

def l6_build_or_load_level6_kb(ctx):
    P = ctx['modules']['P']
    kb_path, manifest_path = l6_kb_paths(SEED_TO_RUN, EVAL_SPLIT)
    identity = dict(
        schema_version=2,
        knowledge_base_name=KNOWLEDGE_BASE_NAME,
        seed=SEED_TO_RUN,
        split=EVAL_SPLIT,
        adapter=L6_ADAPTER_VERSION,
        knowledge_token_count=5,
        target_caption_token_count=2,
        max_relation_token_count=3,
        fallback_policy='caption_fill_no_yolo_no_duplicate_no_fabrication',
        final_config_sha256=P.digest(ctx['final_config']),
        contract_sha256=ctx['run']['contract_sha256'],
        input_provenance_sha256=P.digest(dict(sorted(ctx['inputs'].items()))),
        expected_image_count=len(ctx['ids']),
        image_ids_sha256=l6_ids_sha256(ctx['ids']),
        caption_ranking='Level5A_top50_minmax_mmr_dynamic_k2_to_k5',
        relation_ranking='unique_SPO_minmax_confidence_plus_CLIP_MMR_up_to_k3',
        relation_source='RelTR',
    )

    reusable = False
    if kb_path.is_file() and manifest_path.is_file():
        try:
            manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
            for key, value in identity.items():
                if manifest.get(key) != value:
                    raise ValueError(f'KB manifest mismatch: {key}')
            current_hash = l6_sha256(kb_path)
            if manifest.get('knowledge_base_sha256') != current_hash:
                raise ValueError('KB JSONL hash mismatch')
            records = l6_read_level6_kb(kb_path, ctx['ids'])
            reusable = True
            print('LEVEL6 KNOWLEDGE BASE REUSED:', kb_path)
        except (OSError, ValueError, KeyError, TypeError, json.JSONDecodeError) as exc:
            print('Level6 KB cache is stale/invalid; rebuilding:', exc)

    if not reusable:
        records = []
        temporary = kb_path.with_name(kb_path.name + '.pending')
        with temporary.open('w', encoding='utf-8') as handle:
            for position, iid in enumerate(ctx['ids']):
                row = l6_select_kb_record(ctx, position)
                handle.write(json.dumps(row, ensure_ascii=False, allow_nan=False,
                                        separators=(',', ':')) + '\n')
                records.append(row)
                if (position + 1) % 500 == 0 or position + 1 == len(ctx['ids']):
                    print(f'BUILD LEVEL6 KB: {position + 1}/{len(ctx["ids"])}', flush=True)
        os.replace(temporary, kb_path)

        distribution = {}
        for row in records:
            key = str(row['relation_count'])
            distribution[key] = distribution.get(key, 0) + 1
        kb_hash = l6_sha256(kb_path)
        manifest = dict(
            identity,
            status='COMPLETE',
            knowledge_base_path=str(kb_path),
            knowledge_base_sha256=kb_hash,
            record_count=len(records),
            relation_count_distribution=distribution,
            fallback_image_count=sum(1 for r in records if r['fallback_used']),
            storage='jsonl',
            generated_at=l6_now(),
            note=('Uses up to 3 distinct real RelTR relations per image. Missing relation slots '
                  'are filled by retrieved captions. No YOLO, no relation duplication, no fabricated relation.')
        )
        temporary_manifest = manifest_path.with_name(manifest_path.name + '.pending')
        temporary_manifest.write_text(
            json.dumps(manifest, ensure_ascii=False, indent=2, allow_nan=False),
            encoding='utf-8')
        os.replace(temporary_manifest, manifest_path)
        print('LEVEL6 KNOWLEDGE BASE CREATED:', kb_path)

    records = l6_read_level6_kb(kb_path, ctx['ids'])
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if manifest.get('status') != 'COMPLETE':
        raise ValueError('Level6 KB manifest is not COMPLETE')
    if manifest.get('record_count') != len(ctx['ids']):
        raise ValueError('Level6 KB manifest record_count mismatch')
    if manifest.get('knowledge_base_sha256') != l6_sha256(kb_path):
        raise ValueError('Level6 KB changed after build/reuse')

    distribution = {}
    for row in records:
        key = str(row['relation_count'])
        distribution[key] = distribution.get(key, 0) + 1
    ctx['level6_kb_records'] = records
    ctx['level6_kb_path'] = kb_path
    ctx['level6_kb_manifest_path'] = manifest_path
    ctx['level6_kb_sha256'] = manifest['knowledge_base_sha256']
    ctx['level6_kb_manifest'] = manifest
    ctx['relation_count_distribution'] = distribution
    print('LEVEL6 KB VALIDATED:', len(records), 'images | relation_count_distribution=', distribution)
    return ctx


## Level 6 /7 — Knowledge-only: bỏ đóng góp nhánh thị giác, force `g=1`

Không xóa kiến trúc visual khỏi checkpoint. Chỉ override fusion ở inference để `Context = C_knw`, tương đương visual weight = 0 và knowledge weight = 1. State dict vẫn giữ nguyên để load checkpoint Level 5A strict.


In [15]:
def l6_apply_knowledge_only(model, M):
    before = M.state_dict_sha256(model.state_dict())
    keys = tuple(model.state_dict())
    def force_one(self, v_img, k_tokens, scores):
        if self.training: raise RuntimeError('Knowledge-only is inference-only')
        if not torch.isfinite(v_img).all(): raise ValueError('Non-finite visual input')
        return k_tokens.new_ones((k_tokens.shape[0], 1))
    model.gated_fusion.forward = types.MethodType(force_one, model.gated_fusion)
    model.eval()
    model.requires_grad_(False)
    assert tuple(model.state_dict()) == keys
    assert M.state_dict_sha256(model.state_dict()) == before
    return before

@torch.no_grad()
def l6_assert_visual_zero(model, v, k, scores, bos_id):
    ids = torch.full((1, 1), bos_id, dtype=torch.long, device=v.device)
    received = []
    def observe(layer, args):
        assert torch.equal(args[3], torch.ones_like(args[3]))
        assert args[2].shape[-2:] == (5, model.knowledge_proj.out_features)
        received.append(tuple(args[2].shape))
    handles = [layer.register_forward_pre_hook(observe) for layer in model.layers]
    visual_hooks = []
    try:
        logits, gate = model(v, k, scores, ids)
        alternate, alternate_gate = model(-3 * v, k, scores, ids)
        assert torch.equal(gate, torch.ones_like(gate)) and torch.equal(alternate_gate, gate)
        torch.testing.assert_close(logits, alternate, rtol=0, atol=0)
        def overwrite_visual(module, args, output):
            return (torch.full_like(output[0], 123.0), output[1])
        visual_hooks = [layer.cross_vis.register_forward_hook(overwrite_visual) for layer in model.layers]
        intervened, _ = model(v, k, scores, ids)
        torch.testing.assert_close(logits, intervened, rtol=0, atol=0)
        assert len(received) == 3 * len(model.layers)
        assert torch.isfinite(logits).all()
        return dict(visual_weight=0.0, knowledge_weight=1.0, forced_gate=float(gate.item()),
            visual_invariance_verified=True, cross_knw_shapes=[list(shape) for shape in received],
            max_logit_change=float((logits-intervened).abs().max().item()))
    finally:
        for handle in handles + visual_hooks: handle.remove()

## Level 6 /8 — Strict selected-seed checkpoint loader (no fallback)

`l6_load_model(SEED_TO_RUN)` reads `trained_5tokens_v1/seedXX/best_model.pt` next to the reviewed source tree. It validates training evidence, seed, source hashes, model-state hash and the recorded vocabulary; calls `create_level5(stoi, device)` and `load_state_dict(..., strict=True)`. It never calls `MS.Experiment`, warm-start, or a training phase. Definitions are executed here; loading occurs in /10 after the /9 resume check.


In [16]:
def l6_checkpoint_seed(path):
    matches = [int(x) for x in re.findall(r'(?:^|[/\\])seed(42|43|44|45|46)(?=[/\\]|$)', str(path))]
    return matches[0] if len(matches) == 1 else None

def l6_validate_source_contract(contract):
    for field in ('code_sha256', 'shared_modules_sha256'):
        for name, digest in contract[field].items():
            if name not in SOURCE_PATHS or l6_sha256(SOURCE_PATHS[name]) != digest:
                raise ValueError('Checkpoint/source mismatch: ' + name)

def l6_load_model(seed):
    if seed != SEED_TO_RUN:
        raise ValueError('Only SEED_TO_RUN may be loaded; no cross-seed fallback')
    modules = l6_setup_modules()
    M, S, P = modules['M'], modules['S'], modules['P']
    home = modules['code_dir'].parent / 'trained_5tokens_v1' / f'seed{seed}'
    path = home / 'best_model.pt'
    if not path.is_file():
        raise FileNotFoundError(f'MISSING checkpoint seed{seed}: {path}. No fallback; no training.')
    run = l6_read_json_ro(home / 'run_manifest.json')
    cfg = l6_read_json_ro(home / 'final_config.json')
    assert run['seed'] == cfg['seed'] == seed
    assert run['architecture'] == M.ARCHITECTURE_ID and run['smoke'] is False
    assert run['contract_sha256'] == cfg['contract_sha256']
    assert cfg['selected_on'] == 'validation_only' and cfg['w'] == .6
    assert cfg['lambda_param'] in P.LAMBDA_GRID
    l6_validate_source_contract(run)
    checkpoint = S.load_torch(path)
    contract, evidence, state = checkpoint['contract'], checkpoint['evidence'], checkpoint['model_state_dict']
    assert contract['seed'] == seed and contract['contract_sha256'] == run['contract_sha256']
    l6_validate_source_contract(contract)
    assert evidence['trained'] is True and evidence['training_seed'] == seed
    assert evidence['completed_epochs'] == 15 and evidence['optimizer_steps'] > 0
    state_hash = M.state_dict_sha256(state)
    assert state_hash == evidence['model_state_sha256'] == cfg['checkpoint_sha256']
    assert state_hash != evidence['initial_state_sha256']
    assert all(torch.isfinite(value).all() for value in state.values())
    ctx = dict(modules=modules, seed=seed, home=home, run=run, final_config=cfg,
        checkpoint_path=path, checkpoint_sha256=l6_sha256(path), state_sha256=state_hash,
        evidence=evidence, inputs={str(home/'run_manifest.json'):l6_sha256(home/'run_manifest.json'),
            str(home/'final_config.json'):l6_sha256(home/'final_config.json')},
        data_root=modules['code_dir'].parents[2])
    vocab_path = l6_input(ctx, ['vocab.json'])
    stoi = l6_read_json_ro(vocab_path)['stoi']
    assert sorted(stoi.values()) == list(range(len(stoi)))
    assert all(token in stoi for token in ('<PAD>','<BOS>','<EOS>','<UNK>'))
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    P.set_seed(seed)
    model, _, beam = M.create_level5(stoi, device)
    model.load_state_dict(state, strict=True)
    model.eval()
    assert M.state_dict_sha256(model.state_dict()) == state_hash
    assert (model.knowledge_proj.in_features, model.knowledge_proj.out_features) == (512,256)
    l6_apply_knowledge_only(model, M)
    ctx.update(model=model, beam=beam, stoi=stoi, device=device)
    del checkpoint, state
    l6_log(f'LOADED seed{seed}: {path}; strict=True; gate=1; device={device}')
    return ctx

def l6_input(ctx, names):
    matches = [(Path(path),digest) for path,digest in ctx['run']['input_sha256'].items()
               if Path(path).name in names]
    if len(matches) != 1:
        raise FileNotFoundError(f'Expected exactly one recorded input {names}; found {matches}')
    original, digest = matches[0]
    old_root = Path(ctx['run']['init_checkpoint']).parents[3]
    path = ctx['data_root'] / original.relative_to(old_root)
    if not path.is_file(): raise FileNotFoundError(f'MISSING recorded input: {path} (original {original})')
    if str(path) not in ctx['inputs']:
        l6_log('VERIFY input: ' + str(path))
        if l6_sha256(path) != digest: raise ValueError('Input hash differs from Level5A: ' + str(path))
        ctx['inputs'][str(path)] = digest
    return path


## Level 6 /9 — Validate completed seed and skip / force rerun


In [17]:
L6_COMPLETION_CHECKS = (
    'source_loaded', 'checkpoint_loaded',
    'up_to_3_distinct_real_relations', 'caption_fallback_without_yolo',
    'no_relation_duplication_or_fabrication', 'exactly_5_knowledge_items',
    'shape_matches_implementation', 'visual_contribution_zero',
    'knowledge_contribution_one', 'real_caption_generated',
    'materialized_level6_kb_validated', 'no_training', 'full_split_completed')

def l6_ids_sha256(image_ids):
    normalized = sorted(str(value) for value in image_ids)
    return hashlib.sha256(json.dumps(normalized, ensure_ascii=False,
        separators=(',', ':')).encode('utf-8')).hexdigest()

def l6_check_completed_seed(seed, split, verify_inputs=False):
    """Read-only: validate genuine completed outputs, never infer completion from file names."""
    if seed not in SEEDS:
        return dict(valid=False, reason='invalid seed')
    folder = l6_output_path(f'seed{seed}')
    manifest_path = folder / 'experiment_manifest.json'
    try:
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        if manifest.get('status') != 'COMPLETE' or manifest.get('seed') != seed:
            raise ValueError('manifest is not COMPLETE for this seed')
        expected = dict(
            dataset_name='COCO_2014', ablation_mode='knowledge_only',
            knowledge_token_count=5, target_caption_token_count=2,
            max_relation_token_count=3, relation_fallback='caption_fill_no_yolo',
            force_gate=1.0, visual_weight=0.0, knowledge_weight=1.0,
            split=split, beam_size=3, run_mode='one_seed_full_split',
            sample_limit=None, training=False)
        if any(key not in manifest.get('config', {}) or manifest['config'][key] != value
               for key, value in expected.items()):
            raise ValueError('config/split/composition does not match')
        if manifest['config'].get('seed') != seed:
            raise ValueError('config seed does not match')
        if manifest.get('knowledge_base_name') != KNOWLEDGE_BASE_NAME:
            raise ValueError('missing/wrong materialized Level6 knowledge base')
        kb_path = manifest.get('knowledge_base_path')
        kb_hash = manifest.get('knowledge_base_sha256')
        if not kb_path or not kb_hash or not Path(kb_path).is_file() or l6_sha256(kb_path) != kb_hash:
            raise ValueError('materialized Level6 knowledge base missing or hash mismatch')
        if not all(manifest.get('checks', {}).get(key) is True for key in L6_COMPLETION_CHECKS):
            raise ValueError('required inference checks not all verified')
        protocol = manifest.get('protocol_sha256')
        if not isinstance(protocol, str) or len(protocol) != 64 or any(c not in '0123456789abcdef' for c in protocol):
            raise ValueError('missing/invalid common protocol hash')
        for name in ('predictions.json', 'metrics.json', 'debug_retrieval.json'):
            expected_hash = manifest.get('artifact_sha256', {}).get(name)
            if not expected_hash or l6_sha256(folder / name) != expected_hash:
                raise ValueError('artifact missing or hash mismatch: ' + name)
        predictions = json.loads((folder / 'predictions.json').read_text(encoding='utf-8'))
        if not isinstance(predictions, list) or not predictions:
            raise ValueError('predictions must be a non-empty list')
        if not all(isinstance(row, dict) and row.get('image_id') is not None
                   and isinstance(row.get('caption'), str) and row['caption'].strip() for row in predictions):
            raise ValueError('invalid prediction records')
        ids = [str(row['image_id']) for row in predictions]
        if len(set(ids)) != len(ids) or len(ids) != manifest.get('expected_image_count'):
            raise ValueError('duplicate/missing images in full split')
        if l6_ids_sha256(ids) != manifest.get('expected_image_ids_sha256'):
            raise ValueError('prediction image IDs do not cover expected split')
        metrics = json.loads((folder / 'metrics.json').read_text(encoding='utf-8'))
        if metrics.get('seed') != seed or metrics.get('split') != split or metrics.get('status') != 'COMPLETE':
            raise ValueError('metrics seed/split/status mismatch')
        required_metrics = manifest.get('metric_names')
        if not isinstance(required_metrics, list) or not required_metrics or len(set(required_metrics)) != len(required_metrics):
            raise ValueError('missing/invalid evaluator metric names')
        for scale in ('metrics_raw', 'metrics_x100'):
            values = metrics.get(scale)
            if not isinstance(values, dict) or set(values) != set(required_metrics):
                raise ValueError('metric keys differ from evaluator protocol')
            if any(isinstance(v, bool) or not isinstance(v, (int, float)) or not math.isfinite(v)
                   for v in values.values()):
                raise ValueError('non-finite/non-numeric metrics')
        if verify_inputs:
            checkpoint = manifest.get('checkpoint_path')
            if not checkpoint or l6_checkpoint_seed(checkpoint) != seed or l6_priority(checkpoint) != 0:
                raise ValueError('checkpoint provenance does not match selected L5A seed')
            if not Path(checkpoint).resolve().is_relative_to(SEARCH_ROOT.resolve()):
                raise ValueError('checkpoint outside allowed search root')
            if l6_sha256(checkpoint) != manifest.get('checkpoint_sha256'):
                raise ValueError('checkpoint changed')
            if not L5_SOURCE_HASHES_BEFORE or manifest.get('source_sha256') != L5_SOURCE_HASHES_BEFORE:
                raise ValueError('source provenance changed/not resolved')
            inputs = manifest.get('input_sha256')
            if not isinstance(inputs, dict) or not inputs:
                raise ValueError('cache/dataset input provenance missing')
            for path, digest in {**manifest['source_sha256'], **inputs}.items():
                if not Path(path).resolve().is_relative_to(SEARCH_ROOT.resolve()):
                    raise ValueError('input outside allowed search root')
                if l6_sha256(path) != digest:
                    raise ValueError('source/cache/split input changed: ' + path)
        return dict(valid=True, reason='verified COMPLETE', manifest=manifest, metrics=metrics)
    except (OSError, ValueError, TypeError, KeyError, AttributeError) as exc:
        return dict(valid=False, reason=str(exc))

existing = l6_check_completed_seed(SEED_TO_RUN, EVAL_SPLIT, verify_inputs=True)
L6_RUN_STATE = dict(seed=SEED_TO_RUN, split=EVAL_SPLIT, status='PENDING', reasons=[])
if existing['valid']:
    print(f'SEED {SEED_TO_RUN} ALREADY COMPLETED')
    if not FORCE_RERUN:
        L6_RUN_STATE['status'] = 'SKIPPED'
        print('SKIP — set FORCE_RERUN=True only to rerun this seed.')
    else:
        print('FORCE RERUN requested for selected seed only; old completed results remain until successful replacement.')
else:
    print('No reusable completed run:', existing['reason'])
print('Selected seed:', SEED_TO_RUN, '| output:', l6_output_path(f'seed{SEED_TO_RUN}'))

No reusable completed run: [Errno 2] No such file or directory: '/content/drive/.shortcut-targets-by-id/1OCkkcQKlIYTCPLnGgb2y4aTf_VJFE7xQ/Data/baseline_level6/_Chi_Dung_TriThuc/seed42/experiment_manifest.json'
Selected seed: 42 | output: /content/drive/.shortcut-targets-by-id/1OCkkcQKlIYTCPLnGgb2y4aTf_VJFE7xQ/Data/baseline_level6/_Chi_Dung_TriThuc/seed42


### COCO_2014-only protocol check

Cell này xác nhận cấu hình dataset trước khi chạy inference. Việc prune `train2014/val2014/test2014` trong `/4` chỉ áp dụng cho **source discovery**, không loại COCO_2014 khỏi thí nghiệm.


In [18]:
assert DATASET_NAME == 'COCO_2014'
assert L6_CONFIG['dataset_name'] == 'COCO_2014'
assert not any(x in DATASET_NAME.casefold() for x in ('2017', 'flickr'))
print('DATASET PROTOCOL: COCO_2014 ONLY')
print('Split:', EVAL_SPLIT)
print('Output:', LEVEL6_DIR)


DATASET PROTOCOL: COCO_2014 ONLY
Split: test
Output: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level6/_Chi_Dung_TriThuc


### Chính sách relation fallback — không dùng YOLO

- Có ≥3 relation RelTR thật: **2 caption + 3 relation**.
- Có 2 relation: **3 caption + 2 relation**.
- Có 1 relation: **4 caption + 1 relation**.
- Có 0 relation: **5 caption**.

Luôn giữ đúng **5 knowledge tokens**. Không duplicate relation, không tạo relation giả, không dùng YOLO.


## Level 6 /10 — Full-split inference, original beam search, resume chunks

`l6_run_inference` evaluates every image in the original selected split with the original `beam_search_single` (k=3, alpha=0.6, max_len=30), under `model.eval()` and `torch.no_grad()`. First 10 images include retrieval, projection shapes and generated captions. Prediction chunks under the selected seed can resume; force rerun uses a new staging folder. No other seed is executed.


Trước khi sinh caption, cell này build/reuse và validate kho tri thức Level 6 vật lý cho seed hiện tại; inference chỉ đọc 5 item đã materialize từ kho này.

**Top-50 cache fallback:** nếu cache Top-50 của Level 5A bị thiếu, notebook tự tính lại Top-50 chính xác bằng FAISS từ các feature/embedding đã được xác minh và chỉ lưu cache mới trong `baseline_level6/_Chi_Dung_TriThuc/cache`. Không ghi vào Level 5A.


In [19]:
def l6_failure(exc):
    message = f'{type(exc).__name__}: {exc}'
    l6_log('FAIL: ' + message)
    L6_RUN_STATE.update(status='FAIL', reasons=[message])
    report = dict(status='FAIL', seed=SEED_TO_RUN, split=EVAL_SPLIT, timestamp=l6_now(), error=message)
    l6_write_json(f'seed{SEED_TO_RUN}/last_attempt.json', report)
    old = l6_seed_output('experiment_manifest.json')
    if not old.is_file() or json.loads(old.read_text(encoding='utf-8')).get('status') != 'COMPLETE':
        l6_write_json(f'seed{SEED_TO_RUN}/experiment_manifest.json', report)

def l6_prepare_data(ctx):
    P, S = ctx['modules']['P'], ctx['modules']['S']
    split = EVAL_SPLIT
    ann_path = l6_assert_coco2014_only_path(
        l6_input(ctx, ['dataset_coco.json', f'captions_karpathy_{split}.json']),
        'COCO_2014 annotation')
    ann = l6_read_json_ro(ann_path)
    if 'images' in ann and 'annotations' not in ann:
        caps = {int(r['cocoid']):[s['raw'].strip() for s in r['sentences'] if s['raw'].strip()]
                for r in ann['images'] if r['split'] == split}
    elif 'annotations' in ann:
        caps = {}
        for row in ann['annotations']:
            caps.setdefault(int(row['image_id']), []).append(row['caption'].strip())
    else:
        caps = {int(k):v if isinstance(v,list) else [v] for k,v in ann.items()}
    ids = sorted(caps)
    l6_log(f'DATASET CONTRACT OK: {DATASET_NAME} | split={split} | images={len(ids)} | annotation={ann_path}')
    assert len(ids) == P.COUNTS[split] and sum(map(len,caps.values())) == 25010
    assert all(caps[i] and all(isinstance(c,str) and c.strip() for c in caps[i]) for i in ids)
    feature_path = l6_assert_coco2014_only_path(
        l6_input(ctx,[f'{split}_features.npy',f'karpathy_{split}_features.npy']),
        'COCO_2014 feature cache')
    feature_ids_path = l6_assert_coco2014_only_path(
        l6_input(ctx,[f'{split}_image_ids.json',f'karpathy_{split}_image_ids.json']),
        'COCO_2014 feature-id cache')
    features = np.load(feature_path, mmap_mode='r')
    feature_ids = list(map(int,l6_read_json_ro(feature_ids_path)))
    assert features.shape == (len(ids),512) and len(set(feature_ids)) == len(ids) and set(feature_ids) == set(ids)
    assert np.isfinite(features).all() and (np.linalg.norm(features,axis=1)>0).all()
    reltr_path = l6_assert_coco2014_only_path(
        l6_input(ctx,[f'coco_{split}_reltr_detection.csv']),
        'COCO_2014 RelTR')
    relations, labels = l6_parse_reltr(reltr_path, ids)
    relation_count_distribution = {k: 0 for k in range(4)}
    for iid in ids:
        relation_count_distribution[min(3, len(relations[iid]))] += 1
    l6_write_json(f'seed{SEED_TO_RUN}/relation_preflight.json', dict(
        split=split, policy='up_to_3_real_relations_then_caption_fill',
        relation_count_distribution=relation_count_distribution,
        fallback_image_count=sum(v for k,v in relation_count_distribution.items() if k < 3),
        no_yolo=True, no_relation_duplication=True, no_relation_fabrication=True))
    l6_log('RELATION PREFLIGHT: ' + repr(relation_count_distribution))
    kb_path = l6_assert_coco2014_only_path(
        l6_input(ctx,['text_embeddings_train_restval.npy']),
        'COCO_2014 caption KB')
    kb = np.load(kb_path,mmap_mode='r')
    assert kb.shape == (566747,512)
    captions, kb_ids = [], []
    with l6_input(ctx,['text_embedding_mapping.jsonl']).open(encoding='utf-8') as handle:
        for i,line in enumerate(handle):
            row=json.loads(line)
            assert int(row.get('row_id',i)) == i and row['split'] in ('train','restval')
            assert str(row['caption']).strip()
            captions.append(str(row['caption']).strip()); kb_ids.append(int(row['image_id']))
    assert len(captions)==len(kb) and not set(kb_ids).intersection(ids)
    # ------------------------------------------------------------------
    # Top-50 caption retrieval cache
    #
    # Ưu tiên:
    #   1) reuse READ-ONLY cache Level5A nếu cache tương thích thực sự tồn tại;
    #   2) reuse cache Level6 đã tự tạo;
    #   3) nếu cả hai không có -> tính Top-50 chính xác bằng FAISS từ
    #      COCO_2014 image features + Level5A caption embeddings.
    #
    # Tuyệt đối không ghi vào baseline_level5.
    # ------------------------------------------------------------------
    top_rows, top_scores = [], []
    preferred_l5_cache_root = (
        ctx['home'].parent / 'cache' /
        ctx['run']['contract_sha256'][:20] / 'top50' / split
    )

    feature_digest = ctx['inputs'][str(feature_path)]
    kb_digest = ctx['inputs'][str(kb_path)]
    top50_identity = dict(
        schema_version=1,
        dataset='COCO_2014',
        split=split,
        topk=50,
        metric='cosine_inner_product',
        image_features_sha256=feature_digest,
        caption_embeddings_sha256=kb_digest,
        adapter=L6_ADAPTER_VERSION,
    )
    top50_key = P.digest(top50_identity)
    l6_top50_root = l6_output_path(
        f'cache/top50_coco2014/{split}/{top50_key[:20]}'
    )
    l6_top50_root.mkdir(parents=True, exist_ok=True)

    feature_pos = {iid: pos for pos, iid in enumerate(feature_ids)}
    if len(feature_pos) != len(feature_ids):
        raise ValueError('Duplicate image id in COCO_2014 feature-id cache')

    faiss_index = None

    def l6_load_generated_top50(path, batch):
        path = Path(path)
        if not path.is_file():
            return None
        try:
            with np.load(path, allow_pickle=False) as archive:
                meta = json.loads(str(archive['metadata_json'].item()))
                image_ids_saved = archive['image_ids']
                rows_saved = archive['row_indices']
                scores_saved = archive['clip_scores']
                payload_sha = str(archive['payload_sha256'].item())
            expected_meta = dict(
                schema_version=1,
                identity_sha256=top50_key,
                split=split,
                ids_sha256=P.digest(batch),
                topk=50,
                metric='cosine_inner_product',
            )
            if any(meta.get(k) != v for k, v in expected_meta.items()):
                raise ValueError('Level6 Top50 metadata mismatch')
            if image_ids_saved.tolist() != batch:
                raise ValueError('Level6 Top50 image IDs mismatch')
            payload = hashlib.sha256()
            payload.update(np.ascontiguousarray(image_ids_saved).tobytes())
            payload.update(np.ascontiguousarray(rows_saved).tobytes())
            payload.update(np.ascontiguousarray(scores_saved).tobytes())
            if payload.hexdigest() != payload_sha:
                raise ValueError('Level6 Top50 payload hash mismatch')
            return (
                rows_saved.astype(np.int64, copy=False),
                scores_saved.astype(np.float32, copy=False),
                path,
            )
        except (OSError, ValueError, KeyError, EOFError, zipfile.BadZipFile, json.JSONDecodeError) as exc:
            print('Level6 Top50 cache invalid, rebuilding:', path, '|', exc)
            return None

    def l6_save_generated_top50(path, batch, rows_saved, scores_saved):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        image_ids_saved = np.asarray(batch, dtype=np.int64)
        rows_saved = np.asarray(rows_saved, dtype=np.int64)
        scores_saved = np.asarray(scores_saved, dtype=np.float32)
        payload = hashlib.sha256()
        payload.update(np.ascontiguousarray(image_ids_saved).tobytes())
        payload.update(np.ascontiguousarray(rows_saved).tobytes())
        payload.update(np.ascontiguousarray(scores_saved).tobytes())
        meta = dict(
            schema_version=1,
            identity_sha256=top50_key,
            split=split,
            ids_sha256=P.digest(batch),
            topk=50,
            metric='cosine_inner_product',
            source='recomputed_in_level6_from_verified_COCO2014_features_and_Level5A_caption_embeddings',
            no_write_to_level5=True,
        )
        temporary = path.with_name(path.name + '.pending')
        with temporary.open('wb') as handle:
            np.savez_compressed(
                handle,
                metadata_json=np.asarray(json.dumps(meta, sort_keys=True)),
                image_ids=image_ids_saved,
                row_indices=rows_saved,
                clip_scores=scores_saved,
                payload_sha256=np.asarray(payload.hexdigest()),
            )
        os.replace(temporary, path)
        return path

    def l6_get_faiss_index():
        nonlocal faiss_index
        if faiss_index is not None:
            return faiss_index
        try:
            import faiss
        except Exception as exc:
            raise RuntimeError(
                'faiss-cpu is required to rebuild missing Top-50 cache'
            ) from exc

        dim = int(kb.shape[1])
        index = faiss.IndexFlatIP(dim)

        print(
            'LEVEL5A Top50 cache unavailable -> building exact Level6 FAISS index '
            f'from {len(kb)} caption embeddings ...',
            flush=True
        )
        for begin in range(0, len(kb), 65536):
            block = np.asarray(kb[begin:begin+65536], dtype=np.float32).copy()
            if not np.isfinite(block).all():
                raise ValueError('Non-finite caption embedding while building FAISS Top50')
            faiss.normalize_L2(block)
            index.add(block)
            if begin == 0 or begin + len(block) == len(kb):
                print(f'  FAISS add: {begin + len(block)}/{len(kb)}', flush=True)

        if index.ntotal != len(kb):
            raise RuntimeError(f'FAISS index size mismatch: {index.ntotal} != {len(kb)}')
        faiss_index = index
        return faiss_index

    def l6_compute_top50(batch):
        import faiss
        query = np.asarray(
            [features[feature_pos[iid]] for iid in batch],
            dtype=np.float32
        ).copy()
        if query.shape != (len(batch), 512) or not np.isfinite(query).all():
            raise ValueError('Invalid COCO_2014 query feature batch for Top50 rebuild')
        faiss.normalize_L2(query)
        scores_saved, rows_saved = l6_get_faiss_index().search(query, 50)
        if rows_saved.shape != (len(batch), 50) or scores_saved.shape != (len(batch), 50):
            raise RuntimeError('FAISS Top50 output shape mismatch')
        if (rows_saved < 0).any() or (rows_saved >= len(kb)).any():
            raise RuntimeError('FAISS returned invalid caption row')
        if not np.isfinite(scores_saved).all():
            raise RuntimeError('FAISS returned non-finite similarity score')
        if not all(len(set(map(int, row))) == 50 for row in rows_saved):
            raise RuntimeError('FAISS Top50 contains duplicate caption rows')
        return rows_saved.astype(np.int64), scores_saved.astype(np.float32)

    l5_cache_parent = ctx['home'].parent / 'cache'

    for start in range(0, len(ids), 1000):
        batch = ids[start:start+1000]
        expected_l5 = dict(
            contract_sha256=ctx['run']['contract_sha256'],
            stage='faiss_top50',
            split=split,
            ids_sha256=P.digest(batch),
        )

        rows = scores = actual = None
        l5_errors = []

        # 1) Preferred Level5A cache path.
        l5_candidates = [preferred_l5_cache_root / f'{start:06d}.npz']

        # 2) Search other Level5A contract directories, but accept only if
        #    metadata passes l6_read_cache_ro(expected_l5).
        if l5_cache_parent.is_dir():
            for candidate in sorted(l5_cache_parent.glob(
                f'*/top50/{split}/{start:06d}.npz'
            )):
                if candidate not in l5_candidates:
                    l5_candidates.append(candidate)

        for candidate in l5_candidates:
            try:
                arrays, meta, actual_candidate = l6_read_cache_ro(
                    candidate, expected_l5, S
                )
                rows = arrays['row_indices']
                scores = arrays['clip_scores']
                actual = actual_candidate
                print('REUSE Level5A Top50:', actual, flush=True)
                break
            except RuntimeError as exc:
                l5_errors.append(str(exc))

        # 3) If no compatible L5 cache, reuse/build a Level6 cache.
        if rows is None:
            generated_path = l6_top50_root / f'{start:06d}.npz'
            loaded = l6_load_generated_top50(generated_path, batch)
            if loaded is not None:
                rows, scores, actual = loaded
                print('REUSE Level6 Top50:', actual, flush=True)
            else:
                print(
                    f'BUILD Level6 Top50 batch {start:06d}; '
                    'no compatible Level5A cache found.',
                    flush=True
                )
                rows, scores = l6_compute_top50(batch)
                actual = l6_save_generated_top50(
                    generated_path, batch, rows, scores
                )
                print('SAVED Level6 Top50:', actual, flush=True)

        assert rows.shape == scores.shape == (len(batch), 50)
        assert np.issubdtype(rows.dtype, np.integer)
        assert ((rows >= 0) & (rows < len(kb))).all()
        assert np.isfinite(scores).all()
        assert all(len(set(map(int, r))) == 50 for r in rows)

        # Hash only the cache actually used. Level5A remains READ ONLY.
        ctx['inputs'][str(actual)] = l6_sha256(actual)
        top_rows.append(np.asarray(rows, dtype=np.int64))
        top_scores.append(np.asarray(scores, dtype=np.float32))

    l6_log(
        'TOP50 READY: '
        f'{len(ids)} COCO_2014 {split} images; '
        'compatible Level5A cache reused when available, otherwise Level6 cache used.'
    )
    metadata_path=kb_path.parent/'metadata.json'
    if not metadata_path.is_file(): raise FileNotFoundError('MISSING encoder metadata: '+str(metadata_path))
    ctx['inputs'][str(metadata_path)]=l6_sha256(metadata_path)
    encoder,tokenizer,metadata=l6_load_text_encoder(metadata_path,ctx['device'])
    # Prove that the relation encoder reproduces the existing KB pipeline before using it.
    probe_rows=top_rows[0][0,:3]
    probe=l6_encode_texts([captions[int(i)] for i in probe_rows],encoder,tokenizer,ctx['device'])
    expected=np.asarray(kb[probe_rows],dtype=np.float32)
    assert np.isfinite(expected).all()
    assert np.allclose(np.linalg.norm(expected,axis=1),1,atol=1e-3)
    similarity=(probe*expected).sum(axis=1)
    if not (similarity>0.9999).all():
        raise ValueError('Text encoder does not reproduce Level5A KB embeddings: '+repr(similarity.tolist()))
    texts=sorted({r['serialized_text'] for pool in relations.values() for r in pool})
    embedding_key=P.digest(dict(texts=texts,metadata_sha256=ctx['inputs'][str(metadata_path)],
                              cleaning='nfkc_whitespace_v1',adapter=L6_ADAPTER_VERSION))
    relative=f'cache/relations_{split}_{embedding_key}.npz'
    vector_path=l6_output_path(relative)
    if vector_path.is_file():
        with np.load(vector_path,allow_pickle=False) as archive:
            vectors=archive['vectors'];saved_key=str(archive['key'].item())
            checksum=str(archive['vectors_sha256'].item())
        assert saved_key==embedding_key and hashlib.sha256(vectors.tobytes()).hexdigest()==checksum
    else:
        vectors=np.concatenate([l6_encode_texts(texts[start:start+128],encoder,tokenizer,ctx['device'])
                                for start in range(0,len(texts),128)])
        temporary=vector_path.with_name(vector_path.name+'.pending')
        with temporary.open('wb') as handle:
            np.savez_compressed(handle,vectors=vectors,key=np.asarray(embedding_key),
                                vectors_sha256=np.asarray(hashlib.sha256(vectors.tobytes()).hexdigest()))
        os.replace(temporary,vector_path)
    assert vectors.shape==(len(texts),512) and np.isfinite(vectors).all()
    assert np.allclose(np.linalg.norm(vectors,axis=1),1,atol=1e-3)
    ctx['inputs'][str(vector_path)]=l6_sha256(vector_path)
    ctx.update(ids=ids,caps=caps,features=features,fmap={i:p for p,i in enumerate(feature_ids)},
        relations=relations,labels=labels,kb=kb,kb_captions=captions,
        top_rows=np.concatenate(top_rows),top_scores=np.concatenate(top_scores),
        relation_vectors=vectors,text_index={t:i for i,t in enumerate(texts)},encoder_metadata=metadata,
        encoder_check_cosine=similarity.tolist(), relation_preflight_distribution=relation_count_distribution)
    del encoder,tokenizer
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return ctx

def l6_retrieve(ctx,position):
    if 'level6_kb_records' not in ctx:
        raise RuntimeError('Level6 materialized knowledge base is not loaded')
    iid = int(ctx['ids'][position])
    record = ctx['level6_kb_records'][position]
    l6_validate_kb_record(record, iid)

    caption_rows = np.asarray([int(x['kb_row']) for x in record['captions']], dtype=np.int64)
    caption_embeddings = np.asarray(ctx['kb'][caption_rows], dtype=np.float32)

    if record['relations']:
        relation_embeddings = np.asarray([
            ctx['relation_vectors'][ctx['text_index'][x['serialized_text']]]
            for x in record['relations']
        ], dtype=np.float32)
    else:
        relation_embeddings = np.empty((0, 512), dtype=np.float32)

    expected_caption_count = int(record['caption_count'])
    expected_relation_count = int(record['relation_count'])
    if caption_embeddings.shape != (expected_caption_count, 512):
        raise ValueError(f'image_id={iid}: caption embedding shape mismatch')
    if relation_embeddings.shape != (expected_relation_count, 512):
        raise ValueError(f'image_id={iid}: relation embedding shape mismatch')
    if expected_caption_count + expected_relation_count != 5:
        raise ValueError(f'image_id={iid}: knowledge composition does not total 5')

    k, memory = l6_build_memory(caption_embeddings, relation_embeddings, ctx['model'])
    score_values = ([float(x['score']) for x in record['captions']] +
                    [float(x['score']) for x in record['relations']])
    if len(score_values) != 5:
        raise ValueError(f'image_id={iid}: relevance score count must be 5')
    scores = torch.tensor(score_values, dtype=torch.float32,
                          device=ctx['device']).unsqueeze(0)
    feature = np.asarray(ctx['features'][ctx['fmap'][iid]], dtype=np.float32)
    v = torch.from_numpy(feature.copy()).to(ctx['device']).unsqueeze(0)

    debug = dict(
        image_id=iid,
        selected_captions=record['captions'],
        selected_relations=record['relations'],
        caption_count=record['caption_count'],
        relation_count=record['relation_count'],
        fallback_used=record['fallback_used'],
        fallback_policy=record['fallback_policy'],
        final_knowledge_items=record['knowledge_items'],
        knowledge_types=record['knowledge_types'],
        knowledge_base_path=str(ctx['level6_kb_path']),
        knowledge_base_sha256=ctx['level6_kb_sha256'],
        shape_before_projection=list(k.shape),
        shape_after_projection=list(memory.shape),
        visual_weight=0.0,
        knowledge_weight=1.0,
        relevance_scores=scores.cpu().tolist(),
        caption_label_fallback=bool(record.get('caption_label_fallback', False)),
    )
    return v,k,scores,debug

def l6_run_inference():
    ctx=l6_prepare_data(l6_load_model(SEED_TO_RUN))
    ctx=l6_build_or_load_level6_kb(ctx)
    P=ctx['modules']['P']; model=ctx['model']
    protocol=dict(adapter=L6_ADAPTER_VERSION,split=EVAL_SPLIT,source_hashes=REVIEWED_SOURCE_HASHES,
        composition_policy='up_to_3_distinct_real_relations_then_caption_fill_to_5',gate=1.0,beam=dict(k=3,alpha=.6,max_length=30),
        knowledge_base_name=KNOWLEDGE_BASE_NAME,knowledge_base_sha256=ctx['level6_kb_sha256'],
        knowledge_base_manifest_sha256=l6_sha256(ctx['level6_kb_manifest_path']),
        caption_ranking='original_top50_minmax_mmr_dynamic_k2_to_k5_using_seed_validation_config',
        relation_ranking='canonical_unique_spo; minmax(0.5*minmax(confidence)+0.5*minmax(CLIP)); original_MMR_up_to_k3',
        insufficient_relations='caption_fallback_no_yolo_no_duplicate_no_fabrication',relation_count_distribution=ctx['relation_count_distribution'],encoder=ctx['encoder_metadata'],
        evaluator='original_coco_eval_worker_all_metrics',eval_ids_sha256=l6_ids_sha256(ctx['ids']),
        shared_input_hashes={path:digest for path,digest in ctx['inputs'].items()
                            if 'baseline_level5' not in Path(path).parts})
    protocol_hash=P.digest(protocol)
    run_key=P.digest(dict(protocol=protocol_hash,seed=SEED_TO_RUN,checkpoint=ctx['state_sha256'],
                          inputs=ctx['inputs'],config=ctx['final_config']))
    if FORCE_RERUN: run_key += '_'+datetime.now().strftime('%Y%m%dT%H%M%S%f')
    stage=f'seed{SEED_TO_RUN}/staging/{run_key}'
    l6_output_path(stage).mkdir(parents=True,exist_ok=True)
    ctx.update(stage=stage,protocol=protocol,protocol_sha256=protocol_hash)
    records,debug_rows=[],[]
    verified_gate_calls=[0]
    def guard(layer,args):
        assert not torch.is_grad_enabled() and not layer.training
        assert args[2].shape==(1,5,256) and torch.equal(args[3],torch.ones_like(args[3]))
        verified_gate_calls[0]+=1
    # Verify a real input before any partial prediction cache is accepted.
    with torch.no_grad():
        v,k,scores,first=l6_retrieve(ctx,0)
        ctx['visual_validation']=l6_assert_visual_zero(model,v,k,scores,ctx['stoi']['<BOS>'])
        handles=[layer.register_forward_pre_hook(guard) for layer in model.layers]
        try:
            for start in range(0,len(ctx['ids']),32):
                batch_ids=ctx['ids'][start:start+32]
                relative=f'{stage}/chunks/{start:06d}.json'
                path=l6_output_path(relative)
                if path.is_file():
                    saved=json.loads(path.read_text(encoding='utf-8'))
                    payload={key:value for key,value in saved.items() if key!='payload_sha256'}
                    assert saved['payload_sha256']==P.digest(payload) and saved['run_key']==run_key
                    batch=saved['predictions'];debug=saved['debug']
                    assert [r['image_id'] for r in batch]==batch_ids
                    assert all(isinstance(r['caption'],str) and r['caption'].strip() for r in batch)
                else:
                    batch,debug=[],[]
                    for pos in range(start,start+len(batch_ids)):
                        v,k,scores,item=l6_retrieve(ctx,pos)
                        caption=ctx['beam'](model,v,k,scores,beam_size=3,max_len=30,alpha=.6)
                        if not isinstance(caption,str) or not caption.strip():
                            raise ValueError(f'Beam returned empty caption: image {ctx["ids"][pos]}')
                        batch.append(dict(image_id=ctx['ids'][pos],caption=caption))
                        if pos<10:
                            item['generated_caption']=caption;debug.append(item)
                    payload=dict(run_key=run_key,predictions=batch,debug=debug)
                    l6_write_json(relative,dict(payload,payload_sha256=P.digest(payload)))
                records.extend(batch);debug_rows.extend(debug)
                l6_log(f'seed{SEED_TO_RUN}: {len(records)}/{len(ctx["ids"])} images')
        finally:
            for handle in handles:handle.remove()
    assert [r['image_id'] for r in records]==ctx['ids'] and len(debug_rows)==10
    assert [r['image_id'] for r in debug_rows]==ctx['ids'][:10]
    l6_assert_source_unchanged()
    assert ctx['modules']['M'].state_dict_sha256(model.state_dict())==ctx['state_sha256']
    l6_write_json(f'{stage}/predictions.json',records)
    l6_write_json(f'{stage}/debug_retrieval.json',debug_rows)
    ctx.update(records=records,debug=debug_rows,gate_calls=verified_gate_calls[0])
    L6_RUN_STATE.update(status='INFERENCE_COMPLETE',seed=SEED_TO_RUN,split=EVAL_SPLIT)
    l6_log('FULL SPLIT INFERENCE COMPLETE; evaluation pending')
    return ctx

if not L6_RUN_STATE or L6_RUN_STATE['seed']!=SEED_TO_RUN or L6_RUN_STATE['split']!=EVAL_SPLIT:
    raise RuntimeError('Run cells /3 through /9 for this selected seed first')
if L6_RUN_STATE['status']=='SKIPPED':
    print(f'SEED {SEED_TO_RUN} ALREADY COMPLETED — SKIP')
else:
    try:
        L6_CONTEXT=l6_run_inference()
    except Exception as exc:
        l6_failure(exc)
        raise


Ignoring 1 noncritical dataset scan error(s) during module setup.
VERIFY input: /content/drive/.shortcut-targets-by-id/1OCkkcQKlIYTCPLnGgb2y4aTf_VJFE7xQ/Data/baseline_level1/vocab.json
LOADED seed42: /content/drive/.shortcut-targets-by-id/1OCkkcQKlIYTCPLnGgb2y4aTf_VJFE7xQ/Data/baseline_level5/lv5_A/trained_5tokens_v1/seed42/best_model.pt; strict=True; gate=1; device=cuda
VERIFY input: /content/drive/.shortcut-targets-by-id/1OCkkcQKlIYTCPLnGgb2y4aTf_VJFE7xQ/Data/COCO_2014/annotations_trainval2014/annotations/captions_karpathy_test.json
DATASET CONTRACT OK: COCO_2014 | split=test | images=5000 | annotation=/content/drive/.shortcut-targets-by-id/1OCkkcQKlIYTCPLnGgb2y4aTf_VJFE7xQ/Data/COCO_2014/annotations_trainval2014/annotations/captions_karpathy_test.json
VERIFY input: /content/drive/.shortcut-targets-by-id/1OCkkcQKlIYTCPLnGgb2y4aTf_VJFE7xQ/Data/baseline_level1/clip_image_features/test_features.npy
VERIFY input: /content/drive/.shortcut-targets-by-id/1OCkkcQKlIYTCPLnGgb2y4aTf_VJFE7xQ/Da

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

BUILD LEVEL6 KB: 500/5000
BUILD LEVEL6 KB: 1000/5000
BUILD LEVEL6 KB: 1500/5000
BUILD LEVEL6 KB: 2000/5000
BUILD LEVEL6 KB: 2500/5000
BUILD LEVEL6 KB: 3000/5000
BUILD LEVEL6 KB: 3500/5000
BUILD LEVEL6 KB: 4000/5000
BUILD LEVEL6 KB: 4500/5000
BUILD LEVEL6 KB: 5000/5000
LEVEL6 KNOWLEDGE BASE CREATED: /content/drive/.shortcut-targets-by-id/1OCkkcQKlIYTCPLnGgb2y4aTf_VJFE7xQ/Data/baseline_level6/_Chi_Dung_TriThuc/knowledge_base_level6/seed42/test_max3relation_caption_fallback.jsonl
LEVEL6 KB VALIDATED: 5000 images | relation_count_distribution= {'3': 4464, '1': 175, '2': 234, '0': 127}
seed42: 32/5000 images
seed42: 64/5000 images
seed42: 96/5000 images
seed42: 128/5000 images
seed42: 160/5000 images
seed42: 192/5000 images
seed42: 224/5000 images
seed42: 256/5000 images
seed42: 288/5000 images
seed42: 320/5000 images
seed42: 352/5000 images
seed42: 384/5000 images
seed42: 416/5000 images
seed42: 448/5000 images
seed42: 480/5000 images
seed42: 512/5000 images
seed42: 544/5000 images
seed42:

## Level 6 /11 — Original COCO evaluator and publish completed seed

Calls `level5a_pipeline.Experiment.evaluate` without constructing the experiment, which launches the unchanged `coco_eval_worker.py`: CIDEr, BLEU 1–4, ROUGE-L, METEOR and SPICE. The selected seed is marked COMPLETE only after full coverage and finite real metrics are verified. A failed force rerun keeps previously completed canonical results until publication succeeds.


Protocol dataset: **COCO_2014 only**; no COCO_2017/Flickr evaluation is mixed into this run.


In [20]:
def l6_evaluate_and_publish(ctx):
    import shutil
    assert ctx['seed']==SEED_TO_RUN and L6_RUN_STATE['status']=='INFERENCE_COMPLETE'
    assert L6_RUN_STATE['split']==EVAL_SPLIT==L6_CONFIG['split'] and L6_CONFIG['seed']==SEED_TO_RUN
    assert [r['image_id'] for r in ctx['records']]==ctx['ids']
    # Match the Java 11 evaluator environment used by the original Colab notebook.
    java=Path('/usr/lib/jvm/java-11-openjdk-amd64/bin/java')
    if not java.is_file():
        subprocess.run(['apt-get','update','-qq'],check=True)
        subprocess.run(['apt-get','install','-y','-qq','openjdk-11-jre-headless'],check=True)
    if not java.is_file():raise FileNotFoundError('Java 11 not installed: '+str(java))
    os.environ['JAVA_HOME']=str(java.parent.parent)
    os.environ['PATH']=str(java.parent)+os.pathsep+os.environ['PATH']
    P=ctx['modules']['P']
    evaluator=types.SimpleNamespace(ids={EVAL_SPLIT:ctx['ids']},caps={EVAL_SPLIT:ctx['caps']},smoke=False,
        environment={'pycocoevalcap':importlib.metadata.version('pycocoevalcap')},
        code_hashes={'coco_eval_worker.py':REVIEWED_SOURCE_HASHES['coco_eval_worker.py']},
        cfg={'eval_timeout_seconds':7200})
    output=l6_output_path(ctx['stage']+'/coco_metrics_raw.json')
    raw=P.Experiment.evaluate(evaluator,ctx['records'],EVAL_SPLIT,output,only_cider=False)
    assert set(raw)==set(P.METRICS)
    assert all(not isinstance(v,bool) and isinstance(v,(int,float)) and math.isfinite(v) for v in raw.values())
    metrics=dict(status='COMPLETE',seed=SEED_TO_RUN,split=EVAL_SPLIT,
        evaluator='pycocoevalcap_official_java',metrics_raw=raw,metrics_x100={k:v*100 for k,v in raw.items()},
        evaluated_image_count=len(ctx['ids']),timestamp=l6_now())
    l6_write_json(ctx['stage']+'/metrics.json',metrics)
    l6_assert_source_unchanged()
    if l6_sha256(ctx['checkpoint_path'])!=ctx['checkpoint_sha256']:
        raise ValueError('Checkpoint changed during inference')
    artifacts={name:l6_sha256(l6_output_path(ctx['stage']+'/'+name))
               for name in ('predictions.json','metrics.json','debug_retrieval.json')}
    manifest=dict(status='COMPLETE',seed=SEED_TO_RUN,config=L6_CONFIG,adapter=L6_ADAPTER_VERSION,
        expected_image_count=len(ctx['ids']),expected_image_ids_sha256=l6_ids_sha256(ctx['ids']),
        checks={name:True for name in L6_COMPLETION_CHECKS},protocol_sha256=ctx['protocol_sha256'],
        protocol=ctx['protocol'],source_sha256=L5_SOURCE_HASHES_BEFORE,input_sha256=ctx['inputs'],
        checkpoint_path=str(ctx['checkpoint_path']),checkpoint_sha256=ctx['checkpoint_sha256'],
        checkpoint_state_sha256=ctx['state_sha256'],checkpoint_evidence=ctx['evidence'],
        level5_final_config=ctx['final_config'],metric_names=list(P.METRICS),artifact_sha256=artifacts,
        visual_validation=ctx['visual_validation'],encoder_check_cosine=ctx['encoder_check_cosine'],
        knowledge_base_name=KNOWLEDGE_BASE_NAME,knowledge_base_path=str(ctx['level6_kb_path']),
        knowledge_base_sha256=ctx['level6_kb_sha256'],
        knowledge_base_manifest_path=str(ctx['level6_kb_manifest_path']),
        knowledge_base_manifest_sha256=l6_sha256(ctx['level6_kb_manifest_path']),
        relation_count_distribution=ctx.get('relation_count_distribution'),
        verified_gate_calls_this_session=ctx['gate_calls'],staging_path=str(l6_output_path(ctx['stage'])),
        training=False,timestamp=l6_now())
    # Publish only after genuine full-split evaluation succeeds; manifest is the last commit marker.
    for name in artifacts:
        source=l6_output_path(ctx['stage']+'/'+name)
        target=l6_seed_output(name); temporary=target.with_name(name+'.pending')
        shutil.copyfile(source,temporary);os.replace(temporary,target)
    l6_write_json(f'seed{SEED_TO_RUN}/experiment_manifest.json',manifest)
    checked=l6_check_completed_seed(SEED_TO_RUN,EVAL_SPLIT,verify_inputs=False)
    assert checked['valid'],checked['reason']
    L6_RUN_STATE['status']='COMPLETE'
    l6_log(f'SEED {SEED_TO_RUN} COMPLETE: '+str(l6_seed_output('')))
    print(json.dumps(metrics,ensure_ascii=False,indent=2))
    return metrics

if L6_RUN_STATE and L6_RUN_STATE['status']=='SKIPPED':
    print(f'SEED {SEED_TO_RUN} ALREADY COMPLETED — evaluation skipped')
elif not L6_CONTEXT or not L6_RUN_STATE or L6_RUN_STATE['status']!='INFERENCE_COMPLETE':
    raise RuntimeError('Full-split inference /10 must succeed for the selected seed before evaluation')
else:
    try:
        L6_METRICS=l6_evaluate_and_publish(L6_CONTEXT)
    except Exception as exc:
        l6_failure(exc)
        raise


Đang chấm COCO Java; log: /content/drive/.shortcut-targets-by-id/1OCkkcQKlIYTCPLnGgb2y4aTf_VJFE7xQ/Data/baseline_level6/_Chi_Dung_TriThuc/seed42/staging/9ea41f48bb7e4a53444321169cd0e7e797a803f3eea454f9609b3ee047e5ae08/coco_metrics_raw.log
SEED 42 COMPLETE: /content/drive/.shortcut-targets-by-id/1OCkkcQKlIYTCPLnGgb2y4aTf_VJFE7xQ/Data/baseline_level6/_Chi_Dung_TriThuc/seed42
{
  "status": "COMPLETE",
  "seed": 42,
  "split": "test",
  "evaluator": "pycocoevalcap_official_java",
  "metrics_raw": {
    "CIDEr": 0.8144373230458868,
    "Bleu_1": 0.6744881722711218,
    "Bleu_2": 0.4892488761330302,
    "Bleu_3": 0.34767592087737115,
    "Bleu_4": 0.25124909014824265,
    "ROUGE_L": 0.4906783461456736,
    "METEOR": 0.22992484500956728,
    "SPICE": 0.15750698119721895
  },
  "metrics_x100": {
    "CIDEr": 81.44373230458868,
    "Bleu_1": 67.44881722711217,
    "Bleu_2": 48.92488761330302,
    "Bleu_3": 34.767592087737114,
    "Bleu_4": 25.124909014824265,
    "ROUGE_L": 49.06783461456736,
 

## Level 6 /12 — Aggregate completed seeds (manual)


In [ ]:
RUN_AGGREGATE = False  # Chỉ bật rồi chạy riêng cell này khi cần tổng hợp.

def l6_aggregate_completed_seeds(split):
    completed, rejected, protocol_hashes = {}, {}, set()
    for seed in SEEDS:  # READ-ONLY results inventory, never calls a model/runner
        checked = l6_check_completed_seed(seed, split, verify_inputs=False)
        if checked['valid']:
            completed[seed] = checked
            protocol_hashes.add(checked['manifest']['protocol_sha256'])
        else:
            rejected[str(seed)] = checked['reason']
    if len(protocol_hashes) > 1:
        raise ValueError('Completed seeds have different protocol hashes; refusing to mix experiments')
    metric_key_sets = {tuple(sorted(item['metrics']['metrics_raw'])) for item in completed.values()}
    if len(metric_key_sets) > 1:
        raise ValueError('Completed seeds use different evaluator metric keys')
    result = dict(status='complete' if len(completed) == len(SEEDS) else 'incomplete',
        dataset_name='COCO_2014', split=split, completed_seeds=sorted(completed), missing_seeds=sorted(set(SEEDS)-set(completed)),
        excluded_reasons=rejected, n=len(completed), std_ddof=1,
        protocol_sha256=next(iter(protocol_hashes), None), timestamp=l6_now(),
        metrics_raw=None, metrics_x100=None,
        comparison_limitation=('L5A uses 5 captions + visual/knowledge; L6 is knowledge-only and uses '
            'up to 3 real RelTR relations, with caption fallback to keep exactly 5 knowledge tokens. '
            'Differences cannot be attributed solely to removing visual context.'))
    if completed:
        for scale in ('metrics_raw', 'metrics_x100'):
            result[scale] = {}
            keys = next(iter(completed.values()))['metrics'][scale]
            for metric in sorted(keys):
                values = [entry['metrics'][scale][metric] for entry in completed.values()]
                result[scale][metric] = dict(n=len(values), mean=statistics.mean(values),
                    sample_std=statistics.stdev(values) if len(values)>1 else None)
    path = l6_write_json(f'summary/completed_seeds_{split}.json', result)
    print(json.dumps(result, ensure_ascii=False, indent=2))
    print('Aggregate output:', path)
    return result

if RUN_AGGREGATE:
    L6_AGGREGATE = l6_aggregate_completed_seeds(EVAL_SPLIT)
else:
    print('Aggregate not run. Enable RUN_AGGREGATE and execute this cell separately when needed.')